# 먹스타 ITDA 본선 - End-to-End Pipeline (보고서 흐름 + DRAG family)

이 노트북은 첨부한 itda_full_pipeline.ipynb의 섹션별 직접 실행 셀 형식을 따르되, 실제 내용은 현재 작업공간의 보고서와 src 코드 흐름에 맞춘 본선 파이프라인이다.

핵심 흐름은 다음 순서를 유지한다.

| 단계 | 흐름 |
|---|---|
| 데이터 | YelpZip 라벨 변환, 밀도 중심 30K 샘플링, 시간순 80/20 split |
| 피처 | SBERT 384d + rating + timestamp = 386d, tag 누수 제외 |
| 그래프 | R-T-R / R-S-R / R-Burst-R(+delta t) / R-U-R, 이후 R-Sim-R boost |
| 기본 모델 | HeteroSAGE, HeteroGAT, HeteroBWGNN, TGATLite, TGATLiteV2 |
| DRAG family | BWGAT, DRAG, DRAGWave, DRAGWave_NoRSR, DRAGWave_TVF |
| 최종 평가 | 현재 실행 결과로 transductive와 inductive를 분리 측정하고 보고서 가중치 앙상블 평가 |

작업환경 가정: Google Colab A100 80GB, Google Drive에 사용자가 직접 올린 yelpzip.csv.


## §0 — GPU 확인 및 환경 진단

In [1]:
# ── [Cell 0] GPU 확인 ──────────────────────────────────────────
import subprocess
try:
    out = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
    print(out)
except Exception as e:
    print('nvidia-smi 미지원 환경:', e)
print('A100/L4/V100이 아니면 [런타임] > [런타임 유형 변경]에서 GPU를 선택하세요.')

Fri May 22 11:47:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## §1 - Google Drive 마운트 + 경로 설정

PROJECT_ROOT 또는 WORK_ROOT만 본인 Google Drive 구조에 맞게 수정하면 된다.

MyDrive/ITDA/본선/먹스타_분석코드/ 아래에 data/raw/yelpzip.csv를 두고 실행한다.


In [2]:
# -- [Cell 1] Google Drive 마운트 + 본선 경로 -------------------------
import sys, os
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/ITDA')
except ImportError:
    IN_COLAB = False
    PROJECT_ROOT = Path('.').resolve()

# Drive 구조가 다르면 WORK_ROOT만 바꾼다.
WORK_ROOT = PROJECT_ROOT / '본선' / '먹스타_분석코드' if IN_COLAB else PROJECT_ROOT
RAW       = WORK_ROOT / 'data' / 'raw'
PROC      = WORK_ROOT / 'data' / 'processed'
GRAPH     = WORK_ROOT / 'data' / 'graphs'
EXT       = WORK_ROOT / 'data' / 'external'
MOD       = WORK_ROOT / 'models'
RES       = WORK_ROOT / 'results'
ARTIFACTS = WORK_ROOT / 'artifacts'
RESULTS_DIR = RES

for p in [RAW, PROC, GRAPH, EXT, MOD, RES, ARTIFACTS]:
    p.mkdir(parents=True, exist_ok=True)

print(f'IN_COLAB={IN_COLAB}  WORK_ROOT={WORK_ROOT}')
print('  RAW      :', RAW)
print('  PROC     :', PROC)
print('  GRAPH    :', GRAPH)
print('  MOD      :', MOD)
print('  RES      :', RES)
print('  ARTIFACTS:', ARTIFACTS)
assert (RAW / 'yelpzip.csv').exists(), f'원본 데이터를 업로드하세요: {RAW / "yelpzip.csv"}'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  WORK_ROOT=/content/drive/MyDrive/ITDA/본선/먹스타_분석코드
  RAW      : /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/raw
  PROC     : /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/processed
  GRAPH    : /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/graphs
  MOD      : /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/models
  RES      : /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/results
  ARTIFACTS: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/artifacts


## §2 — 패키지 설치

In [3]:
# ── [Cell 2] 패키지 설치 ─────────────────────────────────────────
if IN_COLAB:
    !pip install -q torch_geometric==2.7.0
    !pip install -q sentence-transformers==2.7.0
    !pip install -q scipy
print('패키지 설치 완료')

패키지 설치 완료


## §3 — 공통 import / 시드 고정 / 디바이스

In [4]:
# ── [Cell 3] 공통 import + 시드 ─────────────────────────────────
import json, time, copy, random, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

from torch_geometric.data import HeteroData, Data
from torch_geometric.nn import HeteroConv, SAGEConv, GATConv, GCNConv, MessagePassing

from scipy.spatial import cKDTree
import scipy.io as sio
import scipy.sparse as sp

warnings.filterwarnings('ignore')

SEED = 42
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM={p.total_memory/1e9:.1f}GB')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB  VRAM=85.1GB


## §4 — `01_eda_sampling.py` 마이그레이션 (EDA + 밀도 중심 30K 샘플링)

원본 가이드: 10K~50K 노드, 무작위 추출 금지, 시간순 80/20 분할.

**핵심**: 상위 100개 식당 풀 → 활성 유저(≥2 리뷰) → 스팸 13.2% 보존 30K 추출 → 시간순 80/20 분할.

In [5]:
# ── [Cell 4] 01_eda_sampling.py 마이그레이션 ────────────────────
SAMPLED_PATH = PROC / 'df_sampled.parquet'

if SAMPLED_PATH.exists():
    print(f'{SAMPLED_PATH.name} 이미 존재 → SKIP. 로드만 수행.')
    df_sample = pd.read_parquet(SAMPLED_PATH)
    print(f'  shape: {df_sample.shape}')
else:
    print('='*60); print('[1] 데이터 로드 & 라벨 변환'); print('='*60)
    df = pd.read_csv(RAW / 'yelpzip.csv', low_memory=False)
    print(f'원본 shape : {df.shape}')
    print(f'컬럼       : {df.columns.tolist()}')

    print(f"\n원본 label 분포: {dict(Counter(df['label'].tolist()))}")
    df['label'] = df['label'].map({-1: 1, 1: 0})    # 사기=-1→1, 정상=1→0
    print(f"변환 label 분포: {dict(Counter(df['label'].tolist()))}")
    spam_ratio = df['label'].mean()
    print(f'스팸 비율: {spam_ratio:.3f} ({spam_ratio*100:.1f}%)')

    print('\n'+'='*60); print('[2] 기본 EDA'); print('='*60)
    df['date']      = pd.to_datetime(df['date'], errors='coerce')
    df['timestamp'] = df['date'].astype('int64') // 10**9
    df['timestamp'] = df['timestamp'].fillna(0).astype('int64')
    print(f"date 범위: {df['date'].min()} ~ {df['date'].max()}")
    prod_counts = df.groupby('prod_id').size().sort_values(ascending=False)
    print(f'식당 수: {len(prod_counts)}')

    print('\n'+'='*60); print('[3] 시간 분포'); print('='*60)
    df_valid = df[df['date'].notnull()].copy()
    yearly = df_valid.groupby(df_valid['date'].dt.year).agg(
        count=('label','size'), spam_count=('label','sum')
    ).assign(spam_ratio=lambda x: x['spam_count']/x['count'])
    print(yearly.to_string())

    print('\n'+'='*60); print('[4] 밀도 중심 샘플링 (전략 A+B+C)'); print('='*60)
    TARGET_NODES = 30_000

    # Step 1 — 상위 100개 식당 풀
    top_prods = prod_counts.head(100).index.tolist()
    df_step1 = df[df['prod_id'].isin(top_prods)].copy()
    print(f'Step1 상위 100 식당: {len(df_step1):,}, 스팸 {df_step1["label"].mean():.3f}')

    # Step 2 — 활성 유저(≥2 리뷰)
    user_in_top = df_step1.groupby('user_id').size().sort_values(ascending=False)
    active_users = user_in_top[user_in_top >= 2].index.tolist()
    df_step2 = df[df['user_id'].isin(active_users) & df['prod_id'].isin(top_prods)].copy()
    print(f'Step2 활성 유저: {len(df_step2):,}, 스팸 {df_step2["label"].mean():.3f}')

    # Step 3 — 스팸 비율 보존 30K
    n_spam_target   = int(TARGET_NODES * spam_ratio)
    n_normal_target = TARGET_NODES - n_spam_target
    df_spam   = df_step2[df_step2['label']==1].copy()
    df_normal = df_step2[df_step2['label']==0].copy()
    if len(df_spam) < n_spam_target:
        prod_spam_counts = df[df['label']==1].groupby('prod_id').size().sort_values(ascending=False)
        extra_prods = prod_spam_counts.head(150).index.tolist()
        df_spam_extra = df[df['prod_id'].isin(extra_prods) & (df['label']==1)].copy()
        df_spam = pd.concat([df_spam, df_spam_extra]).drop_duplicates()
        print(f'스팸 보충 후: {len(df_spam):,}')

    df_spam_sample   = df_spam.sort_values('date').tail(n_spam_target).copy()
    df_normal_sample = df_normal.sort_values('date').tail(n_normal_target).copy()
    df_sample = pd.concat([df_spam_sample, df_normal_sample]).sort_values('date').reset_index(drop=True)

    print(f'\n최종 샘플: {len(df_sample):,}  스팸 {df_sample["label"].mean():.3f}')
    print(f'식당 {df_sample["prod_id"].nunique()}  유저 {df_sample["user_id"].nunique()}')

    # Step 4 — 시간순 80/20 분할
    print('\n'+'='*60); print('[5] 시간순 80/20 분할'); print('='*60)
    n_sample = len(df_sample)
    train_cutoff = int(n_sample * 0.8)
    df_sample['split'] = 'test'
    df_sample.loc[:train_cutoff-1, 'split'] = 'train'
    train_cut_date = df_sample.iloc[train_cutoff-1]['date']
    print(f'분할 기준일: {train_cut_date.date()}')
    print(f'train: {(df_sample["split"]=="train").sum():,}  test: {(df_sample["split"]=="test").sum():,}')

    df_sample = df_sample.reset_index(drop=True)
    df_sample['node_id'] = df_sample.index

    df_sample.to_parquet(SAMPLED_PATH, index=False)
    print(f'\n저장: {SAMPLED_PATH}')

    eda_summary = {
        '원본_총_리뷰수': int(len(df)),
        '샘플_리뷰수': int(len(df_sample)),
        '스팸_비율_원본': round(float(spam_ratio), 4),
        '스팸_비율_샘플': round(float(df_sample['label'].mean()), 4),
        '식당_수': int(df_sample['prod_id'].nunique()),
        '유저_수': int(df_sample['user_id'].nunique()),
        'train_cut_date': str(train_cut_date.date()),
    }
    with open(RES / 'eda_summary.json', 'w', encoding='utf-8') as f:
        json.dump(eda_summary, f, ensure_ascii=False, indent=2)

print('\n✅ §4 EDA + 샘플링 완료')

df_sampled.parquet 이미 존재 → SKIP. 로드만 수행.
  shape: (30000, 11)

✅ §4 EDA + 샘플링 완료


## §5 — `02_features.py` 마이그레이션 (SBERT + 386d 노드 피처)

보고서 §2.3·§3.2에 명시된 **386차원** 노드 피처 (SBERT 384 + rating 1 + timestamp 1) 그대로.
`tag` 컬럼은 label과 100% 동일하므로 절대 포함하지 않음(정답 누수 방지).

In [6]:
# ── [Cell 5] 02_features.py 마이그레이션 ───────────────────────
!pip install -q transformers==4.38.2 sentence-transformers==2.7.0

import os
FEAT_PATH       = PROC / 'node_features.pt'
SBERT_PATH      = PROC / 'sbert_embeddings.pt'
LABEL_PATH      = PROC / 'labels.pt'
TRAIN_MASK_PATH = PROC / 'train_mask.pt'
TEST_MASK_PATH  = PROC / 'test_mask.pt'
TS_PATH         = PROC / 'timestamps.pt'

if all(p.exists() for p in [FEAT_PATH, SBERT_PATH, LABEL_PATH, TRAIN_MASK_PATH, TEST_MASK_PATH, TS_PATH]):
    print('node_features.pt 등 산출물 이미 존재 → SKIP. 로드만 수행.')
    node_features = torch.load(FEAT_PATH,       map_location='cpu', weights_only=True)
    sbert_emb     = torch.load(SBERT_PATH,      map_location='cpu', weights_only=True)
    labels_t      = torch.load(LABEL_PATH,      map_location='cpu', weights_only=True)
    train_mask    = torch.load(TRAIN_MASK_PATH, map_location='cpu', weights_only=True)
    test_mask     = torch.load(TEST_MASK_PATH,  map_location='cpu', weights_only=True)
    timestamps    = torch.load(TS_PATH,         map_location='cpu', weights_only=True)
    print(f'  node_features: {tuple(node_features.shape)}')
else:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("런타임을 재시작합니다. 재시작 후 이 셀을 다시 실행해주세요.")
        os.kill(os.getpid(), 9)

    df = pd.read_parquet(SAMPLED_PATH)
    print(f'shape: {df.shape}')

    print('\n[SBERT 임베딩 — all-MiniLM-L6-v2]')
    model = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
    texts = df['text'].fillna('').tolist()
    print(f'대상: {len(texts):,}건')
    emb = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    )
    sbert_emb = torch.tensor(emb, dtype=torch.float32)
    torch.save(sbert_emb, SBERT_PATH)
    print(f'SBERT: {tuple(sbert_emb.shape)}')

    # 스칼라 피처
    rating_feat = torch.tensor(((df['rating'].fillna(3.0).values - 1.0) / 4.0),
                               dtype=torch.float32).unsqueeze(1)
    ts = df['timestamp'].values.astype(np.float64)
    ts_min, ts_max = ts.min(), ts.max()
    ts_norm = (ts - ts_min) / (ts_max - ts_min + 1e-8)
    ts_feat = torch.tensor(ts_norm, dtype=torch.float32).unsqueeze(1)

    # 노드 피처 = SBERT(384) + rating(1) + timestamp(1) = 386
    node_features = torch.cat([sbert_emb, rating_feat, ts_feat], dim=1)
    print(f'\n노드 피처: {tuple(node_features.shape)} (SBERT 384 + rating 1 + timestamp 1)')
    torch.save(node_features, FEAT_PATH)

    labels_t   = torch.tensor(df['label'].values, dtype=torch.long)
    train_mask = torch.tensor(df['split'].values == 'train', dtype=torch.bool)
    test_mask  = torch.tensor(df['split'].values == 'test',  dtype=torch.bool)
    timestamps = torch.tensor(df['timestamp'].values, dtype=torch.long)
    torch.save(labels_t,   LABEL_PATH)
    torch.save(train_mask, TRAIN_MASK_PATH)
    torch.save(test_mask,  TEST_MASK_PATH)
    torch.save(timestamps, TS_PATH)
    print(f"\ntrain: {train_mask.sum().item():,}  test: {test_mask.sum().item():,}")
    print(f"spam in train: {labels_t[train_mask].sum().item()}  "
          f"({labels_t[train_mask].float().mean().item():.3f})")
    print(f"spam in test:  {labels_t[test_mask].sum().item()}  "
          f"({labels_t[test_mask].float().mean().item():.3f})")

print('\n✅ §5 피처 생성 완료')


shape: (30000, 11)

[SBERT 임베딩 — all-MiniLM-L6-v2]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

대상: 30,000건


Batches:   0%|          | 0/118 [00:00<?, ?it/s]

SBERT: (30000, 384)

노드 피처: (30000, 386) (SBERT 384 + rating 1 + timestamp 1)

train: 24,000  test: 6,000
spam in train: 3245  (0.135)
spam in test:  721  (0.120)

✅ §5 피처 생성 완료


## §6 — `03_graph_build.py` 마이그레이션 (4종 헤테로 엣지 그래프)

**버그 수정**: 원본 153~160줄에 `tree = cKDTree(ts_vals)` / `pairs = sorted(...)`가 for 루프 바깥으로 빠져 있어 IndentationError 발생. 의도된 위치(루프 안쪽)로 교정.

엣지 4종: R-T-R (같은 식당+같은 月), R-S-R (같은 식당+같은 별점), R-Burst-R (72h 윈도우 + Δt 엣지속성), R-U-R (같은 유저). R-Sim-R은 §11에서 boost 단계에 추가.

In [7]:
# ── [Cell 6] 03_graph_build.py 마이그레이션 (버그 수정 포함) ────
GRAPH_PATH = GRAPH / 'hetero_graph.pt'

if GRAPH_PATH.exists():
    print(f'{GRAPH_PATH.name} 이미 존재 → SKIP. 로드만 수행.')
    data = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
    print(data)
else:
    df = pd.read_parquet(SAMPLED_PATH)
    print(f'노드 수: {len(df):,}  피처 dim: {node_features.shape[1]}')
    assert len(df) == node_features.shape[0]

    print('\n[R-T-R] 동일 prod_id + 동일 연-월')
    df['year_month'] = df['date'].dt.to_period('M').astype(str)
    rtr_src, rtr_dst = [], []
    for (prod, ym), group in df.groupby(['prod_id', 'year_month']):
        nodes = group['node_id'].tolist()
        if len(nodes) < 2:
            continue
        if len(nodes) > 32:
            np.random.seed(42)
            nodes_sampled = np.random.choice(nodes, 32, replace=False).tolist()
        else:
            nodes_sampled = nodes
        for i in range(len(nodes_sampled)):
            for j in range(len(nodes_sampled)):
                if i != j:
                    rtr_src.append(nodes_sampled[i])
                    rtr_dst.append(nodes_sampled[j])
    rtr_edge_index = torch.tensor([rtr_src, rtr_dst], dtype=torch.long)
    print(f'  R-T-R 엣지: {rtr_edge_index.shape[1]:,}')

    print('\n[R-S-R] 동일 prod_id + 동일 rating')
    df['rating_int'] = df['rating'].fillna(3.0).astype(int)
    rsr_src, rsr_dst = [], []
    for (prod, rat), group in df.groupby(['prod_id', 'rating_int']):
        nodes = group['node_id'].tolist()
        if len(nodes) < 2:
            continue
        if len(nodes) > 32:
            np.random.seed(42)
            nodes_sampled = np.random.choice(nodes, 32, replace=False).tolist()
        else:
            nodes_sampled = nodes
        for i in range(len(nodes_sampled)):
            for j in range(len(nodes_sampled)):
                if i != j:
                    rsr_src.append(nodes_sampled[i])
                    rsr_dst.append(nodes_sampled[j])
    rsr_edge_index = torch.tensor([rsr_src, rsr_dst], dtype=torch.long)
    print(f'  R-S-R 엣지: {rsr_edge_index.shape[1]:,}')

    print('\n[R-Burst-R] 동일 prod_id + 72h 이내 (cKDTree, Δt 엣지 피처)')
    BURST_WINDOW_SEC = 72 * 3600
    MAX_EDGES_PER_PROD = 2000
    burst_src, burst_dst, burst_delta_t = [], [], []
    for prod, group in df.groupby('prod_id'):
        nodes = group['node_id'].values
        ts_vals = group['timestamp'].values.astype(np.float64).reshape(-1, 1)
        if len(nodes) < 2:
            continue
        # [BUG FIX] 원본에서 이 두 줄이 for 루프 바깥에 있어 IndentationError 발생
        tree = cKDTree(ts_vals)
        pairs = list(tree.query_pairs(r=BURST_WINDOW_SEC))
        if len(pairs) > MAX_EDGES_PER_PROD:
            # [BUG FIX] 원본 pairs = sorted(...) 줄도 들여쓰기 없이 빠져 있었음
            pairs = sorted(pairs, key=lambda p: abs(ts_vals[p[0], 0] - ts_vals[p[1], 0]))
            pairs = pairs[:MAX_EDGES_PER_PROD]
        for (i, j) in pairs:
            ni, nj = int(nodes[i]), int(nodes[j])
            dt = float(abs(ts_vals[i, 0] - ts_vals[j, 0])) / 3600.0
            burst_src.extend([ni, nj]); burst_dst.extend([nj, ni])
            burst_delta_t.extend([dt, dt])
    burst_edge_index = torch.tensor([burst_src, burst_dst], dtype=torch.long)
    burst_edge_attr  = torch.tensor(burst_delta_t, dtype=torch.float32).unsqueeze(1)
    print(f'  R-Burst-R 엣지: {burst_edge_index.shape[1]:,}')
    print(f'  Δt 범위: {burst_edge_attr.min().item():.1f}h ~ {burst_edge_attr.max().item():.1f}h')

    print('\n[R-U-R] 동일 user_id')
    MAX_EDGES_PER_USER = 20
    rur_src, rur_dst = [], []
    for user, group in df.groupby('user_id'):
        nodes = group['node_id'].tolist()
        if len(nodes) < 2:
            continue
        if len(nodes) > 10:
            sorted_nodes = group.sort_values('date')['node_id'].tolist()
            done = False
            for k in range(len(sorted_nodes)):
                for l in range(k+1, min(k+4, len(sorted_nodes))):
                    rur_src.extend([sorted_nodes[k], sorted_nodes[l]])
                    rur_dst.extend([sorted_nodes[l], sorted_nodes[k]])
                    if len(rur_src) > MAX_EDGES_PER_USER * 2:
                        done = True; break
                if done: break
        else:
            for i in range(len(nodes)):
                for j in range(len(nodes)):
                    if i != j:
                        rur_src.append(nodes[i]); rur_dst.append(nodes[j])
    rur_edge_index = torch.tensor([rur_src, rur_dst], dtype=torch.long)
    print(f'  R-U-R 엣지: {rur_edge_index.shape[1]:,}')

    print('\n[HeteroData 구축]')
    N = len(df)
    data = HeteroData()
    data['review'].x         = node_features
    data['review'].y         = labels_t
    data['review'].timestamp = timestamps
    data['review'].train_mask = train_mask
    data['review'].test_mask  = test_mask
    data['review'].node_id    = torch.arange(N)
    data['review', 'rtr',   'review'].edge_index = rtr_edge_index
    data['review', 'rsr',   'review'].edge_index = rsr_edge_index
    data['review', 'burst', 'review'].edge_index = burst_edge_index
    data['review', 'burst', 'review'].edge_attr  = burst_edge_attr
    data['review', 'rur',   'review'].edge_index = rur_edge_index

    total_edges = sum([
        rtr_edge_index.shape[1], rsr_edge_index.shape[1],
        burst_edge_index.shape[1], rur_edge_index.shape[1],
    ])
    print(f'\n  노드={N:,}  피처={node_features.shape[1]}  총 엣지={total_edges:,}')
    torch.save(data, GRAPH_PATH)
    print(f'저장: {GRAPH_PATH}')

    stats = {
        'nodes': int(N), 'feat_dim': int(node_features.shape[1]),
        'edges_rtr': int(rtr_edge_index.shape[1]),
        'edges_rsr': int(rsr_edge_index.shape[1]),
        'edges_burst': int(burst_edge_index.shape[1]),
        'edges_rur': int(rur_edge_index.shape[1]),
        'total_edges': int(total_edges),
        'train_nodes': int(train_mask.sum()), 'test_nodes': int(test_mask.sum()),
        'spam_ratio': round(float(labels_t.float().mean()), 4),
    }
    with open(RES / 'graph_stats.json', 'w', encoding='utf-8') as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

print('\n✅ §6 그래프 구축 완료')

노드 수: 30,000  피처 dim: 386

[R-T-R] 동일 prod_id + 동일 연-월
  R-T-R 엣지: 464,798

[R-S-R] 동일 prod_id + 동일 rating
  R-S-R 엣지: 317,408

[R-Burst-R] 동일 prod_id + 72h 이내 (cKDTree, Δt 엣지 피처)
  R-Burst-R 엣지: 113,548
  Δt 범위: 0.0h ~ 72.0h

[R-U-R] 동일 user_id
  R-U-R 엣지: 62,968

[HeteroData 구축]

  노드=30,000  피처=386  총 엣지=958,722
저장: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/graphs/hetero_graph.pt

✅ §6 그래프 구축 완료


## §7 — `04_train_baseline.py` 마이그레이션 (모델·학습 유틸·3개 베이스라인)

`FEAT_DIM = 388` 하드코딩 → `data['review'].x.shape[1]`로 동적 추출.
Focal Loss(γ=2, α=0.75), AdamW lr=5e-4, CosineAnnealingLR, 150 epoch, 10-epoch마다 평가, patience=20//10.

In [8]:
# ── [Cell 7] FocalLoss / evaluate / train_model 정의 ─────────────
HIDDEN       = 128
NUM_EPOCHS   = 150
PATIENCE     = 20
LR_BASE      = 5e-4
WEIGHT_DECAY = 1e-5

EDGE_TYPES_4 = [
    ('review', 'rtr',   'review'),
    ('review', 'rsr',   'review'),
    ('review', 'burst', 'review'),
    ('review', 'rur',   'review'),
]
EDGE_TYPES_NO_BURST = [
    ('review', 'rtr',   'review'),
    ('review', 'rsr',   'review'),
    ('review', 'rur',   'review'),
]

class FocalLossBinary(nn.Module):
    """Binary Focal Loss with logits — 보고서 표준값 γ=2, α=0.75"""
    def __init__(self, gamma: float = 2.0, alpha: float = 0.75):
        super().__init__()
        self.gamma, self.alpha = gamma, alpha
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt  = torch.exp(-bce)
        w   = torch.where(targets == 1,
                          torch.full_like(bce, self.alpha),
                          torch.full_like(bce, 1 - self.alpha))
        return (w * (1 - pt) ** self.gamma * bce).mean()

def evaluate_binary(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(data)
        probs  = torch.sigmoid(logits[mask]).cpu().numpy()
        labels = data['review'].y[mask].cpu().numpy()
    pr_auc   = average_precision_score(labels, probs)
    preds    = (probs >= 0.5).astype(int)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {'PR-AUC': round(pr_auc, 4), 'Macro-F1': round(macro_f1, 4), 'probs': probs}

def train_model_v1(model, name, data, epochs=NUM_EPOCHS, patience=PATIENCE, lr=LR_BASE):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = FocalLossBinary(gamma=2.0, alpha=0.75)
    train_mask_t = data['review'].train_mask
    labels = data['review'].y
    best_pr_auc, best_state, no_improve = 0.0, None, 0
    history, t0 = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits[train_mask_t], labels[train_mask_t])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step(); scheduler.step()
        if epoch % 10 == 0 or epoch == 1:
            tr = evaluate_binary(model, data, train_mask_t)
            te = evaluate_binary(model, data, data['review'].test_mask)
            elapsed = time.time() - t0
            print(f"  [{name}] ep={epoch:3d}  loss={loss.item():.4f}  "
                  f"train={tr['PR-AUC']:.4f}  test={te['PR-AUC']:.4f}  ({elapsed:.0f}s)")
            history.append({'epoch': epoch, 'PR-AUC': te['PR-AUC'],
                            'Macro-F1': te['Macro-F1'], 'loss': round(loss.item(), 4)})
            if te['PR-AUC'] > best_pr_auc:
                best_pr_auc = te['PR-AUC']
                best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve  = 0
            else:
                no_improve += 1
                if no_improve >= patience // 10:
                    print(f"  [{name}] Early stop @ {epoch}"); break
    model.load_state_dict(best_state)
    final   = evaluate_binary(model, data, data['review'].test_mask)
    elapsed = round(time.time() - t0, 1)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n  [{name}] FINAL  PR-AUC={final['PR-AUC']:.4f}  "
          f"Macro-F1={final['Macro-F1']:.4f}  params={n_params:,}  time={elapsed}s\n")
    torch.save(best_state, MOD / f'{name}_best.pt')
    pd.DataFrame(history).to_csv(RES / f'history_{name}.csv', index=False)
    return final, n_params, elapsed, history

print('학습 유틸 정의 완료')

학습 유틸 정의 완료


In [9]:
# ── [Cell 8] 4종 엣지용 모델 정의: HeteroSAGE, HeteroGAT, HeteroBWGNN ──
class DualFreqConv(MessagePassing):
    """Low-pass(이웃 평균) + High-pass(자신 - 이웃) 동시 학습"""
    def __init__(self, in_ch, out_ch):
        super().__init__(aggr='mean')
        self.lin = nn.Linear(in_ch * 2, out_ch)
    def forward(self, x, edge_index):
        low  = self.propagate(edge_index, x=x)
        high = x - low
        return self.lin(torch.cat([low, high], dim=-1))
    def message(self, x_j):
        return x_j

def _make_classifier(hidden, dropout):
    return nn.Sequential(
        nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1)
    )

class HeteroSAGE(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.3, edge_types=EDGE_TYPES_4):
        super().__init__()
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1 = HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.conv2 = HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.bn1   = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop  = nn.Dropout(dropout); self.cls = _make_classifier(hidden, dropout)
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        x_dict = {'review': x}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn1(x_dict['review'])))}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn2(x_dict['review'])))}
        return self.cls(x_dict['review']).squeeze(-1)

class HeteroGAT(nn.Module):
    def __init__(self, in_ch, hidden, heads=4, dropout=0.3, edge_types=EDGE_TYPES_4):
        super().__init__()
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1 = HeteroConv(
            {et: GATConv(hidden, hidden // heads, heads=heads,
                         dropout=dropout, add_self_loops=False)
             for et in edge_types}, aggr='sum')
        self.conv2 = HeteroConv(
            {et: GATConv(hidden, hidden // heads, heads=heads,
                         dropout=dropout, add_self_loops=False)
             for et in edge_types}, aggr='sum')
        self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout); self.cls = _make_classifier(hidden, dropout)
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        x_dict = {'review': x}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn1(x_dict['review'])))}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn2(x_dict['review'])))}
        return self.cls(x_dict['review']).squeeze(-1)

class HeteroBWGNN(nn.Module):
    def __init__(self, in_ch, hidden, dropout=0.3, edge_types=EDGE_TYPES_4):
        super().__init__()
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1 = HeteroConv({et: DualFreqConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.conv2 = HeteroConv({et: DualFreqConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout); self.cls = _make_classifier(hidden, dropout)
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        x_dict = {'review': x}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn1(x_dict['review'])))}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn2(x_dict['review'])))}
        return self.cls(x_dict['review']).squeeze(-1)

print('정적 GNN 3종 클래스 정의 완료')

정적 GNN 3종 클래스 정의 완료


In [10]:
# ── [Cell 9] 3개 베이스라인 학습 실행 ───────────────────────────
set_seed(42)
data = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False).to(DEVICE)
FEAT_DIM = data['review'].x.shape[1]
print(f'노드={data["review"].x.shape[0]:,}  FEAT_DIM={FEAT_DIM}')

baseline_results = []
LOG_PATH = RES / 'experiment_log.csv'

models_to_run = [
    ('HeteroSAGE',  HeteroSAGE(FEAT_DIM, HIDDEN)),
    ('HeteroGAT',   HeteroGAT(FEAT_DIM, HIDDEN)),
    ('HeteroBWGNN', HeteroBWGNN(FEAT_DIM, HIDDEN)),
]
for name, model in models_to_run:
    ckpt = MOD / f'{name}_best.pt'
    if ckpt.exists():
        print(f'='*60); print(f'▶ {name} — 체크포인트 발견, 평가만 수행'); print('='*60)
        model = model.to(DEVICE)
        model.load_state_dict(torch.load(ckpt, weights_only=True))
        final = evaluate_binary(model, data, data['review'].test_mask)
        n_params = sum(p.numel() for p in model.parameters())
        elapsed  = 0.0
        print(f"  [{name}] LOAD  PR-AUC={final['PR-AUC']:.4f}  Macro-F1={final['Macro-F1']:.4f}")
    else:
        print('='*60); print(f'▶ {name} 학습 시작'); print('='*60)
        final, n_params, elapsed, _ = train_model_v1(model, name, data)

    baseline_results.append({
        'model': name, 'pr_auc': final['PR-AUC'], 'macro_f1': final['Macro-F1'],
        'params': n_params, 'train_sec': elapsed,
        'notes': '정적 베이스라인, Focal γ=2 α=0.75',
    })

df_results = pd.DataFrame(baseline_results)
df_results.to_csv(LOG_PATH, index=False)
print('\n=== Baseline 결과 ===')
print(df_results[['model','pr_auc','macro_f1','params']].to_string(index=False))
print('\n✅ §7 베이스라인 학습 완료')

노드=30,000  FEAT_DIM=386
▶ HeteroSAGE 학습 시작
  [HeteroSAGE] ep=  1  loss=0.0538  train=0.4422  test=0.1214  (1s)
  [HeteroSAGE] ep= 10  loss=0.0374  train=0.6609  test=0.2297  (1s)
  [HeteroSAGE] ep= 20  loss=0.0315  train=0.7336  test=0.3232  (1s)
  [HeteroSAGE] ep= 30  loss=0.0289  train=0.7669  test=0.3572  (1s)
  [HeteroSAGE] ep= 40  loss=0.0267  train=0.7877  test=0.3580  (2s)
  [HeteroSAGE] ep= 50  loss=0.0249  train=0.7990  test=0.3442  (2s)
  [HeteroSAGE] ep= 60  loss=0.0226  train=0.8096  test=0.3365  (2s)
  [HeteroSAGE] Early stop @ 60

  [HeteroSAGE] FINAL  PR-AUC=0.3580  Macro-F1=0.4965  params=321,537  time=1.9s

▶ HeteroGAT 학습 시작
  [HeteroGAT] ep=  1  loss=0.0544  train=0.1513  test=0.1403  (0s)
  [HeteroGAT] ep= 10  loss=0.0445  train=0.6341  test=0.2468  (0s)
  [HeteroGAT] ep= 20  loss=0.0389  train=0.6703  test=0.2589  (1s)
  [HeteroGAT] ep= 30  loss=0.0355  train=0.6904  test=0.2705  (1s)
  [HeteroGAT] ep= 40  loss=0.0331  train=0.7096  test=0.2886  (2s)
  [HeteroGAT] e

## §8 — `05_train_tgat.py` 마이그레이션 (TGATLite + Bochner 시간 인코딩)

**버그 수정**: 원본 142줄(`x_boosted.scatter_add_(...)`)·196줄(`out = torch.cat(...)`)이 들여쓰기 없이 노출되어 IndentationError. 메서드 내부로 교정.

In [11]:
# ── [Cell 10] BochnerTimeEncoder + TimeAwareConv(버그수정) + TGATLite ──
D_TIME = 64
HEADS  = 4

class BochnerTimeEncoder(nn.Module):
    """φ(Δt) = [cos(ω·Δt), sin(ω·Δt)] (Lee et al. AAAI 2024)"""
    def __init__(self, d_time: int = 64):
        super().__init__()
        self.d_time = d_time
        self.omega  = nn.Parameter(torch.randn(d_time // 2))
    def forward(self, delta_t):
        delta_t = delta_t.squeeze(-1) if delta_t.dim() == 2 else delta_t
        t = delta_t.unsqueeze(-1) * self.omega.unsqueeze(0)
        return torch.cat([torch.cos(t), torch.sin(t)], dim=-1)

class TimeAwareConv(nn.Module):
    """burst 엣지 전용: 시간 임베딩을 소스 노드에 scatter-add 후 GAT 적용
    [BUG FIX] 원본은 scatter_add_ 줄이 메서드 들여쓰기 밖이라 실행 불가
    """
    def __init__(self, in_ch, out_ch, d_time, heads):
        super().__init__()
        self.time_proj = nn.Linear(d_time, in_ch)
        self.gat = GATConv(in_ch, out_ch // heads, heads=heads,
                           dropout=0.3, add_self_loops=False)
        self.in_ch = in_ch
    def forward(self, x, edge_index, time_emb):
        if edge_index.shape[1] == 0:
            return torch.zeros(x.shape[0],
                               self.gat.out_channels * self.gat.heads,
                               device=x.device)
        time_feat = self.time_proj(time_emb)
        src = edge_index[0]
        x_boosted = x.clone()
        x_boosted.scatter_add_(0, src.unsqueeze(-1).expand(-1, self.in_ch), time_feat)
        return self.gat(x_boosted, edge_index)

class TGATLite(nn.Module):
    def __init__(self, in_ch, hidden, d_time=D_TIME, heads=HEADS, dropout=0.3,
                 edge_types_no_burst=EDGE_TYPES_NO_BURST):
        super().__init__()
        self.time_encoder = BochnerTimeEncoder(d_time)
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1_nonburst = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in edge_types_no_burst}, aggr='sum')
        self.time_conv1 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.conv2_nonburst = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in edge_types_no_burst}, aggr='sum')
        self.time_conv2 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.bn1  = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout)
        self.cls  = nn.Sequential(
            nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1)
        )
        self._edge_types_no_burst = edge_types_no_burst

    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        burst_ei = data['review', 'burst', 'review'].edge_index
        delta_t  = data['review', 'burst', 'review'].edge_attr.squeeze(-1) \
                   if hasattr(data['review','burst','review'], 'edge_attr') and \
                      data['review','burst','review'].edge_attr is not None else None
        time_emb = self.time_encoder(delta_t) if (delta_t is not None and burst_ei.shape[1] > 0) \
                   else torch.zeros(0, D_TIME, device=x.device)
        nb_ei = {et: data.edge_index_dict[et] for et in self._edge_types_no_burst}
        x_nb1 = self.conv1_nonburst({'review': x}, nb_ei)
        x_burst1 = self.time_conv1(x, burst_ei, time_emb)
        x1 = self.drop(F.relu(self.bn1(x_nb1['review'] + x_burst1)))
        x_nb2 = self.conv2_nonburst({'review': x1}, nb_ei)
        x_burst2 = self.time_conv2(x1, burst_ei, time_emb)
        x2 = self.drop(F.relu(self.bn2(x_nb2['review'] + x_burst2)))
        # [BUG FIX] 원본의 out 줄도 메서드 들여쓰기 밖에 있었음
        out = torch.cat([x_nb2['review'], x_burst2], dim=-1)
        return self.cls(out).squeeze(-1)

print('TGATLite 정의 완료 (버그 수정 반영)')

TGATLite 정의 완료 (버그 수정 반영)


In [12]:
# ── [Cell 11] TGATLite 학습 실행 ────────────────────────────────
set_seed(42)
print('='*60); print('▶ TGATLite 학습 (Bochner 시간 인코딩)'); print('='*60)
ckpt = MOD / 'TGATLite_best.pt'
model = TGATLite(FEAT_DIM, HIDDEN)
if ckpt.exists():
    model = model.to(DEVICE)
    model.load_state_dict(torch.load(ckpt, weights_only=True))
    final = evaluate_binary(model, data, data['review'].test_mask)
    n_params = sum(p.numel() for p in model.parameters()); elapsed = 0.0
    print(f"  [TGATLite] LOAD  PR-AUC={final['PR-AUC']}  Macro-F1={final['Macro-F1']}")
else:
    final, n_params, elapsed, _ = train_model_v1(model, 'TGATLite', data)

# experiment_log.csv 업데이트
df_log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    'model':'TGATLite','pr_auc':final['PR-AUC'],'macro_f1':final['Macro-F1'],
    'params':n_params,'train_sec':elapsed,
    'notes':'TGATLite: Bochner 시간 인코딩, burst Δt edge feature',
}])
df_log = pd.concat([df_log, new_row], ignore_index=True)
df_log.to_csv(LOG_PATH, index=False)
print('\n✅ §8 TGATLite 학습 완료')

▶ TGATLite 학습 (Bochner 시간 인코딩)
  [TGATLite] ep=  1  loss=0.0665  train=0.3195  test=0.0981  (0s)
  [TGATLite] ep= 10  loss=0.0450  train=0.5312  test=0.1167  (0s)
  [TGATLite] ep= 20  loss=0.0414  train=0.5682  test=0.1242  (1s)
  [TGATLite] ep= 30  loss=0.0375  train=0.6456  test=0.2077  (1s)
  [TGATLite] ep= 40  loss=0.0331  train=0.7035  test=0.2808  (1s)
  [TGATLite] ep= 50  loss=0.0304  train=0.7425  test=0.3350  (1s)
  [TGATLite] ep= 60  loss=0.0290  train=0.7590  test=0.3473  (1s)
  [TGATLite] ep= 70  loss=0.0277  train=0.7711  test=0.3438  (2s)
  [TGATLite] ep= 80  loss=0.0271  train=0.7798  test=0.3463  (2s)
  [TGATLite] Early stop @ 80

  [TGATLite] FINAL  PR-AUC=0.3473  Macro-F1=0.6183  params=314,145  time=1.9s


✅ §8 TGATLite 학습 완료


## §9 — `05b_train_tgat_v2.py` 마이그레이션 (Log-Bochner + Tri-Path)

기여 1 (LogBochner) 단독 측정용 `TGATLite_LogBochner`, 기여 1+2 통합 `TGATLiteV2` 두 모델.

In [13]:
# ── [Cell 12] LogBochner + TGATLiteLogBochner + TGATLiteV2 정의 ──
EDGE_TYPES_USER    = [('review', 'rur', 'review')]
EDGE_TYPES_PRODUCT = [('review', 'rtr', 'review'), ('review', 'rsr', 'review')]

class LogBochnerTimeEncoder(nn.Module):
    """φ(Δt) = [cos(ω·log(1+Δt)), sin(ω·log(1+Δt))]"""
    def __init__(self, d_time: int = 64):
        super().__init__()
        self.omega = nn.Parameter(torch.randn(d_time // 2))
    def forward(self, delta_t):
        delta_t = delta_t.squeeze(-1) if delta_t.dim() == 2 else delta_t
        log_dt = torch.log1p(delta_t)
        t = log_dt.unsqueeze(-1) * self.omega.unsqueeze(0)
        return torch.cat([torch.cos(t), torch.sin(t)], dim=-1)

class TGATLiteLogBochner(nn.Module):
    """기여 1 단독 ablation: Log-Bochner만 교체"""
    def __init__(self, in_ch, hidden, d_time=D_TIME, heads=HEADS, dropout=0.3,
                 edge_types_no_burst=EDGE_TYPES_NO_BURST):
        super().__init__()
        self.time_encoder = LogBochnerTimeEncoder(d_time)
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1_nonburst = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in edge_types_no_burst}, aggr='sum')
        self.time_conv1 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.conv2_nonburst = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in edge_types_no_burst}, aggr='sum')
        self.time_conv2 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Sequential(nn.Linear(hidden*2, 64), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(64, 1))
        self._edge_types_no_burst = edge_types_no_burst
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        burst_ei = data['review','burst','review'].edge_index
        delta_t  = data['review','burst','review'].edge_attr.squeeze(-1)
        time_emb = self.time_encoder(delta_t) if burst_ei.shape[1]>0 \
                   else torch.zeros(0, D_TIME, device=x.device)
        nb_ei = {et: data.edge_index_dict[et] for et in self._edge_types_no_burst}
        x_nb1 = self.conv1_nonburst({'review': x}, nb_ei)
        x_burst1 = self.time_conv1(x, burst_ei, time_emb)
        x1 = self.drop(F.relu(self.bn1(x_nb1['review'] + x_burst1)))
        x_nb2 = self.conv2_nonburst({'review': x1}, nb_ei)
        x_burst2 = self.time_conv2(x1, burst_ei, time_emb)
        return self.cls(torch.cat([x_nb2['review'], x_burst2], dim=-1)).squeeze(-1)

class TGATLiteV2(nn.Module):
    """기여 1+2: Tri-Path Dual Memory + Log-Bochner"""
    def __init__(self, in_ch, hidden, d_time=D_TIME, heads=HEADS, dropout=0.3):
        super().__init__()
        self.time_encoder = LogBochnerTimeEncoder(d_time)
        self.proj = nn.Linear(in_ch, hidden)
        # Path 1: User Memory (rur)
        self.user_conv1 = SAGEConv(hidden, hidden)
        self.user_conv2 = SAGEConv(hidden, hidden)
        # Path 2: Product Memory (rtr+rsr)
        self.product_conv1 = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in EDGE_TYPES_PRODUCT}, aggr='mean')
        self.product_conv2 = HeteroConv(
            {et: SAGEConv(hidden, hidden) for et in EDGE_TYPES_PRODUCT}, aggr='mean')
        # Path 3: Burst Temporal
        self.time_conv1 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.time_conv2 = TimeAwareConv(hidden, hidden, d_time, heads)
        self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Sequential(nn.Linear(hidden*3, 64), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        burst_ei = data['review','burst','review'].edge_index
        delta_t  = data['review','burst','review'].edge_attr.squeeze(-1)
        time_emb = self.time_encoder(delta_t) if burst_ei.shape[1]>0 \
                   else torch.zeros(0, D_TIME, device=x.device)
        rur_ei = data['review','rur','review'].edge_index
        # Layer 1
        x_user1    = self.user_conv1(x, rur_ei)
        x_product1 = self.product_conv1({'review': x},
            {et: data.edge_index_dict[et] for et in EDGE_TYPES_PRODUCT})['review']
        x_burst1   = self.time_conv1(x, burst_ei, time_emb)
        x1 = self.drop(F.relu(self.bn1((x_user1 + x_product1 + x_burst1) / 3.0)))
        # Layer 2
        x_user2    = self.user_conv2(x1, rur_ei)
        x_product2 = self.product_conv2({'review': x1},
            {et: data.edge_index_dict[et] for et in EDGE_TYPES_PRODUCT})['review']
        x_burst2   = self.time_conv2(x1, burst_ei, time_emb)
        out = torch.cat([x_user2, x_product2, x_burst2], dim=-1)
        return self.cls(out).squeeze(-1)

print('LogBochner / TGATLiteLogBochner / TGATLiteV2 정의 완료')

LogBochner / TGATLiteLogBochner / TGATLiteV2 정의 완료


In [14]:
# ── [Cell 13] Ablation 두 모델 학습 ─────────────────────────────
set_seed(42)
ab_results = []
for name, model in [
    ('TGATLite_LogBochner', TGATLiteLogBochner(FEAT_DIM, HIDDEN)),
    ('TGATLiteV2',          TGATLiteV2(FEAT_DIM, HIDDEN)),
]:
    ckpt = MOD / f'{name}_best.pt'
    print('='*60); print(f'▶ {name}'); print('='*60)
    if ckpt.exists():
        model = model.to(DEVICE)
        model.load_state_dict(torch.load(ckpt, weights_only=True))
        final = evaluate_binary(model, data, data['review'].test_mask)
        n_params = sum(p.numel() for p in model.parameters()); elapsed = 0.0
        print(f"  LOAD  PR-AUC={final['PR-AUC']}  Macro-F1={final['Macro-F1']}")
    else:
        final, n_params, elapsed, _ = train_model_v1(model, name, data)
    ab_results.append({'model':name,'pr_auc':final['PR-AUC'],'macro_f1':final['Macro-F1'],
                       'params':n_params,'train_sec':elapsed,
                       'notes':'Ablation (Log-Bochner / Tri-Path)'})

df_log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
df_log = pd.concat([df_log, pd.DataFrame(ab_results)], ignore_index=True)
df_log.to_csv(LOG_PATH, index=False)
print('\n=== Ablation 결과 ===')
print(pd.DataFrame(ab_results)[['model','pr_auc','macro_f1']].to_string(index=False))
print('\n✅ §9 Log-Bochner / TGATLiteV2 학습 완료')

▶ TGATLite_LogBochner
  [TGATLite_LogBochner] ep=  1  loss=0.0586  train=0.4552  test=0.0949  (0s)
  [TGATLite_LogBochner] ep= 10  loss=0.0438  train=0.5336  test=0.1114  (0s)
  [TGATLite_LogBochner] ep= 20  loss=0.0397  train=0.5950  test=0.1439  (1s)
  [TGATLite_LogBochner] ep= 30  loss=0.0348  train=0.6797  test=0.2484  (1s)
  [TGATLite_LogBochner] ep= 40  loss=0.0315  train=0.7318  test=0.3248  (1s)
  [TGATLite_LogBochner] ep= 50  loss=0.0291  train=0.7578  test=0.3454  (1s)
  [TGATLite_LogBochner] ep= 60  loss=0.0279  train=0.7749  test=0.3407  (1s)
  [TGATLite_LogBochner] ep= 70  loss=0.0267  train=0.7839  test=0.3526  (2s)
  [TGATLite_LogBochner] ep= 80  loss=0.0257  train=0.7916  test=0.3872  (2s)
  [TGATLite_LogBochner] ep= 90  loss=0.0251  train=0.8016  test=0.3765  (2s)
  [TGATLite_LogBochner] ep=100  loss=0.0246  train=0.8099  test=0.3696  (2s)
  [TGATLite_LogBochner] Early stop @ 100

  [TGATLite_LogBochner] FINAL  PR-AUC=0.3872  Macro-F1=0.6694  params=314,145  time=2.4s


## §10 — `07_ablation.py` 마이그레이션 (TGATLite-NoTime Ablation)

시간 인코딩 제거의 단독 기여 측정.

In [15]:
# ── [Cell 14] TGATLiteNoTime ablation ───────────────────────────
class TGATLiteNoTime(nn.Module):
    """Bochner 인코딩 제거. burst 엣지도 SAGEConv로만 처리"""
    def __init__(self, in_ch, hidden, dropout=0.3, edge_types=EDGE_TYPES_4):
        super().__init__()
        self.proj = nn.Linear(in_ch, hidden)
        self.conv1 = HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.conv2 = HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr='sum')
        self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout); self.cls = _make_classifier(hidden, dropout)
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x)))
        x_dict = {'review': x}
        x_dict = self.conv1(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn1(x_dict['review'])))}
        x_dict = self.conv2(x_dict, data.edge_index_dict)
        x_dict = {'review': self.drop(F.relu(self.bn2(x_dict['review'])))}
        return self.cls(x_dict['review']).squeeze(-1)

set_seed(42)
print('='*60); print('▶ TGATLite_NoTime (Ablation)'); print('='*60)
ckpt = MOD / 'TGATLite_NoTime_best.pt'
m_nt = TGATLiteNoTime(FEAT_DIM, HIDDEN)
if ckpt.exists():
    m_nt = m_nt.to(DEVICE)
    m_nt.load_state_dict(torch.load(ckpt, weights_only=True))
    final_nt = evaluate_binary(m_nt, data, data['review'].test_mask)
    n_params_nt = sum(p.numel() for p in m_nt.parameters()); elapsed_nt = 0.0
    print(f"  LOAD  PR-AUC={final_nt['PR-AUC']}  Macro-F1={final_nt['Macro-F1']}")
else:
    final_nt, n_params_nt, elapsed_nt, _ = train_model_v1(m_nt, 'TGATLite_NoTime', data)

df_log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
df_log = pd.concat([df_log, pd.DataFrame([{
    'model':'TGATLite_NoTime','pr_auc':final_nt['PR-AUC'],'macro_f1':final_nt['Macro-F1'],
    'params':n_params_nt,'train_sec':elapsed_nt,
    'notes':'Ablation: Bochner 시간 인코딩 제거',
}])], ignore_index=True)
df_log.to_csv(LOG_PATH, index=False)
print('\n✅ §10 NoTime ablation 완료')

▶ TGATLite_NoTime (Ablation)
  [TGATLite_NoTime] ep=  1  loss=0.0538  train=0.4424  test=0.1214  (0s)
  [TGATLite_NoTime] ep= 10  loss=0.0374  train=0.6609  test=0.2297  (0s)
  [TGATLite_NoTime] ep= 20  loss=0.0315  train=0.7336  test=0.3219  (0s)
  [TGATLite_NoTime] ep= 30  loss=0.0289  train=0.7668  test=0.3572  (1s)
  [TGATLite_NoTime] ep= 40  loss=0.0267  train=0.7877  test=0.3578  (1s)
  [TGATLite_NoTime] ep= 50  loss=0.0249  train=0.7989  test=0.3437  (1s)
  [TGATLite_NoTime] ep= 60  loss=0.0226  train=0.8094  test=0.3364  (1s)
  [TGATLite_NoTime] Early stop @ 60

  [TGATLite_NoTime] FINAL  PR-AUC=0.3578  Macro-F1=0.4965  params=321,537  time=1.2s


✅ §10 NoTime ablation 완료


## §11 — `09_boost.py` 마이그레이션 (R-Sim-R 엣지 추가 + Warm Restart + 앙상블)

**보고서의 R-Sim-R boost 단계.**

1. R-Sim-R 엣지 생성 (동일 식당 + SBERT 코사인 유사도 ≥ 0.85, 최대 식당당 3000개)
2. 그래프에 R-Sim-R 추가 → 총 5종 엣지
3. HeteroBWGNN(5종), TGATLite(5종) 두 모델을 LR=2e-4, 400 epoch, CosineAnnealingWarmRestarts (T_0=80, T_mult=2)로 재학습
4. 앙상블 가중치 0.5/0.6/0.7 그리드 평가, 최종 0.6:0.4 채택

DRAG 앙상블용으로 두 부스트 모델의 **test logits**도 본선/artifacts에 저장.


In [16]:
# ── [Cell 15] R-Sim-R 엣지 생성 + 5종 그래프 업데이트 ──────────
EDGE_TYPES_5 = EDGE_TYPES_4 + [('review','sim','review')]
EDGE_TYPES_NO_BURST_5 = EDGE_TYPES_NO_BURST + [('review','sim','review')]

# 매번 깨끗한 그래프에서 시작하기 위해 디스크에서 다시 로드
data5 = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

if ('review','sim','review') in data5.edge_types and \
   data5['review','sim','review'].edge_index.shape[1] > 0:
    print('R-Sim-R 엣지 이미 존재 → 건너뜀')
else:
    df = pd.read_parquet(SAMPLED_PATH)
    emb_np = sbert_emb.numpy().astype(np.float32)
    SIM_THRESHOLD = 0.85
    MAX_SIM_PER_PROD = 3000
    sim_src, sim_dst = [], []
    for prod, grp in df.groupby('prod_id'):
        nids = grp['node_id'].values
        if len(nids) < 2:
            continue
        e = emb_np[nids]
        sim = e @ e.T
        np.fill_diagonal(sim, 0)
        rows, cols = np.where(sim > SIM_THRESHOLD)
        if len(rows) > MAX_SIM_PER_PROD:
            top_idx = np.argsort(sim[rows, cols])[-MAX_SIM_PER_PROD:]
            rows, cols = rows[top_idx], cols[top_idx]
        for r, c in zip(rows, cols):
            sim_src.append(int(nids[r])); sim_dst.append(int(nids[c]))
    sim_edge_index = torch.tensor([sim_src, sim_dst], dtype=torch.long)
    print(f'R-Sim-R 엣지: {sim_edge_index.shape[1]:,}')
    if len(sim_src) > 0:
        sim_spam = df.iloc[sim_src]['label'].mean()
        print(f'  sim 엣지 소스 스팸 비율: {sim_spam:.3f}')
    data5['review','sim','review'].edge_index = sim_edge_index
    torch.save(data5, GRAPH_PATH)
    torch.save(data5, GRAPH / 'hetero_graph_boost.pt')
    print(f'5종 엣지 그래프 저장: {GRAPH_PATH}')
    print('boost 그래프 저장:', GRAPH / 'hetero_graph_boost.pt')

if not (GRAPH / 'hetero_graph_boost.pt').exists():
    torch.save(data5, GRAPH / 'hetero_graph_boost.pt')

data5 = data5.to(DEVICE)
total_edges_5 = sum(data5[et].edge_index.shape[1] for et in EDGE_TYPES_5)
print(f'총 엣지(5종): {total_edges_5:,}')


R-Sim-R 엣지: 4,844
  sim 엣지 소스 스팸 비율: 0.250
5종 엣지 그래프 저장: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/graphs/hetero_graph.pt
boost 그래프 저장: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/graphs/hetero_graph_boost.pt
총 엣지(5종): 963,566


In [17]:
# ── [Cell 16] 5종 엣지용 HeteroBWGNN / TGATLite 정의 + Warm Restart 학습 ──
BOOST_LR = 2e-4
BOOST_EPOCHS = 400

def train_warm_restart(model, name, data, epochs=BOOST_EPOCHS, lr=BOOST_LR,
                       load_prev=False, prev_name=None):
    if load_prev:
        prev_ckpt = MOD / f'{prev_name or name}_best.pt'
        if prev_ckpt.exists():
            sd = torch.load(prev_ckpt, weights_only=True)
            # strict=False: 5종 엣지로 새로 생성된 conv는 freshly init
            model.load_state_dict(sd, strict=False)
            print(f'  이전 가중치 부분 로드: {prev_ckpt.name}')
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=80, T_mult=2, eta_min=1e-6)
    crit = FocalLossBinary(gamma=2.0, alpha=0.75)
    tm, tl = data['review'].train_mask, data['review'].y
    best_pr, best_st, hist = 0.0, None, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        loss = crit(model(data)[tm], tl[tm])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step(ep)
        if ep % 20 == 0 or ep == 1:
            te = evaluate_binary(model, data, data['review'].test_mask)
            lr_now = opt.param_groups[0]['lr']
            print(f"  [{name}] ep={ep:3d}  loss={loss.item():.4f}  "
                  f"PR-AUC={te['PR-AUC']:.4f}  F1={te['Macro-F1']:.4f}  lr={lr_now:.2e}")
            hist.append({'epoch':ep,'PR-AUC':te['PR-AUC'],'Macro-F1':te['Macro-F1'],
                         'loss':round(loss.item(),4)})
            if te['PR-AUC'] > best_pr:
                best_pr = te['PR-AUC']
                best_st = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_st)
    fin = evaluate_binary(model, data, data['review'].test_mask)
    ela = round(time.time() - t0, 1)
    print(f"\n  [{name}] FINAL  PR-AUC={fin['PR-AUC']}  F1={fin['Macro-F1']}  ({ela}s)\n")
    torch.save(best_st, MOD / f'{name}_best.pt')
    pd.DataFrame(hist).to_csv(RES / f'history_{name}_boost.csv', index=False)
    return fin

set_seed(42)
bwgnn_boost = HeteroBWGNN(FEAT_DIM, HIDDEN, edge_types=EDGE_TYPES_5)
tgat_boost  = TGATLite (FEAT_DIM, HIDDEN, edge_types_no_burst=EDGE_TYPES_NO_BURST_5)

ckpt_bw = MOD / 'HeteroBWGNN_boost_best.pt'
ckpt_tg = MOD / 'TGATLite_boost_best.pt'

if ckpt_bw.exists():
    bwgnn_boost = bwgnn_boost.to(DEVICE)
    bwgnn_boost.load_state_dict(torch.load(ckpt_bw, weights_only=True))
    bwgnn_res = evaluate_binary(bwgnn_boost, data5, data5['review'].test_mask)
    print(f"[BWGNN_boost LOAD] PR-AUC={bwgnn_res['PR-AUC']}  F1={bwgnn_res['Macro-F1']}")
else:
    print('='*55); print('[BWGNN_boost] Warm Restart 학습'); print('='*55)
    bwgnn_res = train_warm_restart(bwgnn_boost, 'HeteroBWGNN_boost', data5)

if ckpt_tg.exists():
    tgat_boost = tgat_boost.to(DEVICE)
    tgat_boost.load_state_dict(torch.load(ckpt_tg, weights_only=True))
    tgat_res = evaluate_binary(tgat_boost, data5, data5['review'].test_mask)
    print(f"[TGATLite_boost LOAD] PR-AUC={tgat_res['PR-AUC']}  F1={tgat_res['Macro-F1']}")
else:
    print('='*55); print('[TGATLite_boost] Warm Restart 학습'); print('='*55)
    tgat_res = train_warm_restart(tgat_boost, 'TGATLite_boost', data5)

[BWGNN_boost] Warm Restart 학습
  [HeteroBWGNN_boost] ep=  1  loss=0.0532  PR-AUC=0.1135  F1=0.4680  lr=2.00e-04
  [HeteroBWGNN_boost] ep= 20  loss=0.0400  PR-AUC=0.2263  F1=0.4680  lr=1.71e-04
  [HeteroBWGNN_boost] ep= 40  loss=0.0336  PR-AUC=0.3267  F1=0.4916  lr=1.01e-04
  [HeteroBWGNN_boost] ep= 60  loss=0.0318  PR-AUC=0.3614  F1=0.6158  lr=3.01e-05
  [HeteroBWGNN_boost] ep= 80  loss=0.0312  PR-AUC=0.3646  F1=0.6280  lr=2.00e-04
  [HeteroBWGNN_boost] ep=100  loss=0.0275  PR-AUC=0.3940  F1=0.6666  lr=1.92e-04
  [HeteroBWGNN_boost] ep=120  loss=0.0242  PR-AUC=0.4080  F1=0.6642  lr=1.71e-04
  [HeteroBWGNN_boost] ep=140  loss=0.0205  PR-AUC=0.4187  F1=0.6725  lr=1.39e-04
  [HeteroBWGNN_boost] ep=160  loss=0.0173  PR-AUC=0.4195  F1=0.6774  lr=1.01e-04
  [HeteroBWGNN_boost] ep=180  loss=0.0157  PR-AUC=0.4229  F1=0.6815  lr=6.24e-05
  [HeteroBWGNN_boost] ep=200  loss=0.0146  PR-AUC=0.4207  F1=0.6850  lr=3.01e-05
  [HeteroBWGNN_boost] ep=220  loss=0.0143  PR-AUC=0.4225  F1=0.6836  lr=8.57e-0

In [18]:
# ── [Cell 17] 앙상블 + 최종 결과 + DRAG용 logits 저장 ──────────
test_mask_t = data5['review'].test_mask
labels_test = data5['review'].y[test_mask_t].cpu().numpy()

bwgnn_probs = bwgnn_res['probs']
tgat_probs  = tgat_res['probs']

print('[앙상블 그리드 — BWGNN×w + TGATLite×(1-w)]')
for w_b in [0.5, 0.6, 0.7]:
    w_t = 1.0 - w_b
    ens = bwgnn_probs * w_b + tgat_probs * w_t
    pr = average_precision_score(labels_test, ens)
    f1 = f1_score(labels_test, ens >= 0.5, average='macro', zero_division=0)
    print(f'  BWGNN×{w_b} + TGAT×{w_t:.1f}: PR-AUC={pr:.4f}  F1={f1:.4f}')

ens_probs = bwgnn_probs * 0.6 + tgat_probs * 0.4
ens_pr = average_precision_score(labels_test, ens_probs)
ens_f1 = f1_score(labels_test, ens_probs >= 0.5, average='macro', zero_division=0)

print('\n' + '='*55); print('최종 결과 요약'); print('='*55)
log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame()
boost_rows = [
    {'model':'HeteroBWGNN_boost','pr_auc':bwgnn_res['PR-AUC'],'macro_f1':bwgnn_res['Macro-F1'],
     'params':sum(p.numel() for p in bwgnn_boost.parameters()),'train_sec':0,
     'notes':'R-Sim-R 추가, Warm Restart 400ep'},
    {'model':'TGATLite_boost','pr_auc':tgat_res['PR-AUC'],'macro_f1':tgat_res['Macro-F1'],
     'params':sum(p.numel() for p in tgat_boost.parameters()),'train_sec':0,
     'notes':'R-Sim-R 추가, Warm Restart 400ep'},
    {'model':'Ensemble_BWGNN_TGAT','pr_auc':round(ens_pr,4),'macro_f1':round(ens_f1,4),
     'params':0,'train_sec':0,'notes':'BWGNN×0.6 + TGATLite×0.4'},
]
pd.concat([log, pd.DataFrame(boost_rows)]).to_csv(RES / 'experiment_log_final.csv', index=False)
for r in boost_rows:
    print(f"  {r['model']:22s}: PR-AUC={r['pr_auc']}  F1={r['macro_f1']}")

# DRAG 섹션에서 앙상블 그리드 서치에 쓰기 위해 raw logits을 본선/artifacts에 저장
# evaluate_binary가 sigmoid 확률을 반환하므로 logit으로 역변환
def probs_to_logits_binary(probs):
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    logit_pos = np.log(p / (1 - p))
    return np.stack([-logit_pos, logit_pos], axis=-1)   # [N, 2]
bwgnn_logits_2c = torch.tensor(probs_to_logits_binary(bwgnn_probs), dtype=torch.float32)
tgat_logits_2c  = torch.tensor(probs_to_logits_binary(tgat_probs),  dtype=torch.float32)
torch.save(bwgnn_logits_2c, ARTIFACTS / 'bwgnn_boost_test_logits.pt')
torch.save(tgat_logits_2c,  ARTIFACTS / 'tgatlite_boost_test_logits.pt')
print(f'\nDRAG 앙상블용 logits 저장 ({ARTIFACTS})')
print('\n§11 R-Sim-R boost + 2-way 앙상블 완료')


[앙상블 그리드 — BWGNN×w + TGATLite×(1-w)]
  BWGNN×0.5 + TGAT×0.5: PR-AUC=0.4820  F1=0.7075
  BWGNN×0.6 + TGAT×0.4: PR-AUC=0.4846  F1=0.7141
  BWGNN×0.7 + TGAT×0.3: PR-AUC=0.4854  F1=0.7152

최종 결과 요약
  HeteroBWGNN_boost     : PR-AUC=0.4727  F1=0.7162
  TGATLite_boost        : PR-AUC=0.424  F1=0.6627
  Ensemble_BWGNN_TGAT   : PR-AUC=0.4846  F1=0.7141

DRAG 앙상블용 logits 저장 (/content/drive/MyDrive/ITDA/본선/먹스타_분석코드/artifacts)

§11 R-Sim-R boost + 2-way 앙상블 완료


## §12 — `06_eval_inductive.py` 마이그레이션 (인덕티브 재평가)

test 노드끼리만 연결된 서브그래프로 추론 → 실배포 시뮬레이션.

In [19]:
# ── [Cell 18] 인덕티브 평가 (Test-only 엣지 마스킹) ─────────────
def mask_to_test_only(data: HeteroData, test_mask, edge_types):
    """양 끝점이 모두 test 노드인 엣지만 유지"""
    out = copy.deepcopy(data)
    for et in edge_types:
        ei = data[et].edge_index
        src, dst = ei[0], ei[1]
        m = test_mask[src] & test_mask[dst]
        out[et].edge_index = ei[:, m]
        if hasattr(data[et], 'edge_attr') and data[et].edge_attr is not None:
            out[et].edge_attr = data[et].edge_attr[m]
    return out

print('='*60); print('인덕티브 평가 — Test-only 엣지 마스킹'); print('='*60)

test_mask_full = data['review'].test_mask
data_test_only = mask_to_test_only(data, test_mask_full, EDGE_TYPES_4).to(DEVICE)

print('\n[엣지 수 비교 — 4종 엣지 모델]')
for et in EDGE_TYPES_4:
    full_n = data[et].edge_index.shape[1]
    test_n = data_test_only[et].edge_index.shape[1]
    print(f"  {et[1]:8s} 전체={full_n:>7,}  test-only={test_n:>7,}  ({test_n/max(full_n,1)*100:.1f}%)")

inductive_rows = []
for name, model_inst in [
    ('HeteroSAGE',  HeteroSAGE(FEAT_DIM, HIDDEN)),
    ('HeteroGAT',   HeteroGAT(FEAT_DIM, HIDDEN)),
    ('HeteroBWGNN', HeteroBWGNN(FEAT_DIM, HIDDEN)),
    ('TGATLite',    TGATLite(FEAT_DIM, HIDDEN)),
]:
    pt = MOD / f'{name}_best.pt'
    if not pt.exists():
        print(f'  {name}: 모델 파일 없음, 스킵'); continue
    model_inst.load_state_dict(torch.load(pt, weights_only=True))
    model_inst = model_inst.to(DEVICE)
    m = evaluate_binary(model_inst, data_test_only, test_mask_full.to(DEVICE))
    inductive_rows.append({'model':name,'pr_auc':m['PR-AUC'],'macro_f1':m['Macro-F1'],
                           'params':sum(p.numel() for p in model_inst.parameters()),
                           'train_sec':0,'notes':'test-only 엣지 인덕티브 평가'})
    print(f"  {name:12s}: PR-AUC={m['PR-AUC']:.4f}  Macro-F1={m['Macro-F1']:.4f}")

pd.DataFrame(inductive_rows).to_csv(RES / 'experiment_log_inductive.csv', index=False)
print('\n✅ §12 인덕티브 평가 완료')

인덕티브 평가 — Test-only 엣지 마스킹

[엣지 수 비교 — 4종 엣지 모델]
  rtr      전체=464,798  test-only= 86,638  (18.6%)
  rsr      전체=317,408  test-only= 12,270  (3.9%)
  burst    전체=113,548  test-only= 24,480  (21.6%)
  rur      전체= 62,968  test-only=  7,432  (11.8%)
  HeteroSAGE  : PR-AUC=0.3889  Macro-F1=0.4902
  HeteroGAT   : PR-AUC=0.3648  Macro-F1=0.6538
  HeteroBWGNN : PR-AUC=0.3414  Macro-F1=0.6321
  TGATLite    : PR-AUC=0.3537  Macro-F1=0.6179

✅ §12 인덕티브 평가 완료


## §13 — `10_yelpchi.py` 마이그레이션 (외부 데이터 검증 — 선택)

`data/external/YelpChi.mat` 가 있을 때만 실행. 없으면 자동 스킵.

In [20]:
# ── [Cell 19] YelpChi 외부 검증 (선택적) ────────────────────────
YELPCHI_PATH = EXT / 'YelpChi.mat'
if not YELPCHI_PATH.exists():
    print(f'YelpChi.mat 없음 → §13 SKIP. 외부 검증이 필요하면 {YELPCHI_PATH} 에 업로드.')
else:
    print('='*55); print('YelpChi 외부 검증'); print('='*55)
    mat = sio.loadmat(YELPCHI_PATH)
    features = mat['features']
    if sp.issparse(features):
        features = features.toarray()
    features = features.astype(np.float32)
    chi_labels = mat['label'].flatten()
    uniq = np.unique(chi_labels)
    if -1 in uniq:
        chi_labels = (chi_labels == -1).astype(int)
    elif set(uniq) == {1, 2}:
        chi_labels = (chi_labels == 2).astype(int)
    print(f'노드={features.shape[0]:,}  피처={features.shape[1]}  '
          f'스팸={chi_labels.mean():.3f}')

    edge_srcs, edge_dsts = [], []
    for et_name in ['net_rur','net_rtr','net_rsr']:
        adj = mat[et_name]
        if not sp.issparse(adj):
            adj = sp.csr_matrix(adj)
        adj = adj.tocoo()
        edge_srcs.extend(adj.row.tolist()); edge_dsts.extend(adj.col.tolist())
        print(f'  {et_name}: {adj.nnz:,}')
    edge_index = torch.tensor([edge_srcs, edge_dsts], dtype=torch.long)

    x = torch.tensor(features, dtype=torch.float32)
    y = torch.tensor(chi_labels, dtype=torch.long)
    tr_idx, te_idx = train_test_split(np.arange(len(y)), test_size=0.2,
                                       random_state=42, stratify=y)
    train_m = torch.zeros(len(y), dtype=torch.bool); train_m[tr_idx]=True
    test_m  = torch.zeros(len(y), dtype=torch.bool); test_m[te_idx]=True
    chi_data = Data(x=x, y=y, edge_index=edge_index,
                    train_mask=train_m, test_mask=test_m).to(DEVICE)

    class BWGNN_Chi(nn.Module):
        def __init__(self, in_ch, hidden=128, dropout=0.3):
            super().__init__()
            self.proj = nn.Linear(in_ch, hidden)
            self.conv1 = DualFreqConv(hidden, hidden)
            self.conv2 = DualFreqConv(hidden, hidden)
            self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden)
            self.drop = nn.Dropout(dropout); self.cls = _make_classifier(hidden, dropout)
        def forward(self, data):
            x = self.drop(F.relu(self.proj(data.x)))
            x = self.drop(F.relu(self.bn1(self.conv1(x, data.edge_index))))
            x = self.drop(F.relu(self.bn2(self.conv2(x, data.edge_index))))
            return self.cls(x).squeeze(-1)

    set_seed(42)
    chi_model = BWGNN_Chi(features.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(chi_model.parameters(), lr=2e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=80, T_mult=2)
    crit = FocalLossBinary()
    best_pr, best_st = 0.0, None
    for ep in range(1, 201):
        chi_model.train(); opt.zero_grad()
        loss = crit(chi_model(chi_data)[chi_data.train_mask], chi_data.y[chi_data.train_mask])
        loss.backward(); torch.nn.utils.clip_grad_norm_(chi_model.parameters(), 1.0)
        opt.step(); sch.step(ep)
        if ep % 40 == 0 or ep == 200:
            chi_model.eval()
            with torch.no_grad():
                pr_ = torch.sigmoid(chi_model(chi_data)[chi_data.test_mask]).cpu().numpy()
                la_ = chi_data.y[chi_data.test_mask].cpu().numpy()
            pr_auc = average_precision_score(la_, pr_)
            f1 = f1_score(la_, pr_>=0.5, average='macro', zero_division=0)
            print(f'  ep={ep:3d} loss={loss.item():.4f} PR-AUC={pr_auc:.4f} F1={f1:.4f}')
            if pr_auc > best_pr:
                best_pr = pr_auc
                best_st = {k:v.cpu().clone() for k,v in chi_model.state_dict().items()}
    chi_model.load_state_dict(best_st)
    chi_model.eval()
    with torch.no_grad():
        pr_ = torch.sigmoid(chi_model(chi_data)[chi_data.test_mask]).cpu().numpy()
        la_ = chi_data.y[chi_data.test_mask].cpu().numpy()
    final_pr = average_precision_score(la_, pr_)
    final_f1 = f1_score(la_, pr_>=0.5, average='macro', zero_division=0)
    print(f'\n[YelpChi 결과] PR-AUC={final_pr:.4f}  Macro-F1={final_f1:.4f}')
    torch.save(best_st, MOD / 'BWGNN_YelpChi.pt')
    pd.DataFrame([{'dataset':'YelpChi','model':'BWGNN_Chi',
                   'pr_auc':round(final_pr,4),'macro_f1':round(final_f1,4),
                   'notes':'외부 데이터 검증'}]).to_csv(RES/'yelpchi_result.csv', index=False)
print('\n✅ §13 외부 검증 단계 종료')

YelpChi.mat 없음 → §13 SKIP. 외부 검증이 필요하면 /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/external/YelpChi.mat 에 업로드.

✅ §13 외부 검증 단계 종료


# Part B - 보고서 기준 DRAG family 파이프라인

첨부 노트북의 기존 DRAG 블록은 다른 구현을 사용하므로 여기서 교체한다. 이 파트는 로컬 코드의 다음 흐름을 직접 셀로 옮긴다.

- src/15_drag_bwgat.py: DRAG, BWGAT, DRAGWave 구조
- src/14_new_hypotheses.py: Temporal Velocity Feature(TVF)
- src/27_future_directions.py: DRAGWave_NoRSR, DRAGWave_TVF_400ep
- src/29_performance_boost.py, src/32_ensemble_4way.py: final ensemble 및 transductive/inductive 분리

학습 선택 기준도 원본 코드와 동일하게 test PR-AUC를 체크포인트 선택에 사용한다. 이는 보고서가 명시한 test 참조 흐름을 보존하기 위한 것이다.

## §14 - DRAG family 설정 + boost 그래프 준비


In [21]:
# -- [Cell 20] DRAG family 설정 + boost 그래프 -------------------------
from dataclasses import dataclass

@dataclass
class DragFamilyConfig:
    hidden_dim: int = 128
    heads: int = 4
    dropout: float = 0.3
    lr: float = 5e-4
    weight_decay: float = 1e-5
    focal_gamma: float = 2.0
    focal_alpha: float = 0.75
    base_epochs: int = 150
    final_epochs: int = 400
    eval_every_base: int = 10
    eval_every_final: int = 40
    patience_checks: int = 7
    grad_clip: float = 1.0
    seed: int = 42

DRAG_CFG = DragFamilyConfig()
DRAG_EDGE_TYPES_FULL = EDGE_TYPES_5
DRAG_EDGE_TYPES_NORSR = [('review','rtr','review'),('review','burst','review'),('review','rur','review'),('review','sim','review')]
BOOST_GRAPH_PATH = GRAPH / 'hetero_graph_boost.pt'
if not BOOST_GRAPH_PATH.exists():
    torch.save(data5.cpu(), BOOST_GRAPH_PATH)
data_boost_cpu = torch.load(BOOST_GRAPH_PATH, map_location='cpu', weights_only=False)
data_boost = data_boost_cpu.to(DEVICE)
feat_boost = data_boost['review'].x.shape[1]
print('DRAG config:', DRAG_CFG)
print('boost graph:', BOOST_GRAPH_PATH)
print('edge types:', list(data_boost.edge_types))
print(f"nodes={data_boost['review'].x.shape[0]:,} feat_dim={feat_boost} train={data_boost['review'].train_mask.sum().item():,} test={data_boost['review'].test_mask.sum().item():,}")


DRAG config: DragFamilyConfig(hidden_dim=128, heads=4, dropout=0.3, lr=0.0005, weight_decay=1e-05, focal_gamma=2.0, focal_alpha=0.75, base_epochs=150, final_epochs=400, eval_every_base=10, eval_every_final=40, patience_checks=7, grad_clip=1.0, seed=42)
boost graph: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/data/graphs/hetero_graph_boost.pt
edge types: [('review', 'rtr', 'review'), ('review', 'rsr', 'review'), ('review', 'burst', 'review'), ('review', 'rur', 'review'), ('review', 'sim', 'review')]
nodes=30,000 feat_dim=386 train=24,000 test=6,000


## §15 - 로컬 구현 기준 DRAG / BWGAT / DRAGWave 클래스

DRAG는 관계별 표현을 dynamic relation attention으로 집계하고, BWGAT는 attention-weighted low/high-pass를 쓰며, DRAGWave는 둘을 결합한다.


In [22]:
# -- [Cell 21] DRAG / BWGAT / DRAGWave 정의 --------------------------
class DRAGConv(nn.Module):
    def __init__(self, in_ch, out_ch, n_relations, dropout=0.3):
        super().__init__(); self.rel_convs = nn.ModuleList([SAGEConv(in_ch, out_ch) for _ in range(n_relations)]); self.self_lin = nn.Linear(in_ch, out_ch); self.attn_vec = nn.Linear(out_ch * 2, 1, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x, edge_index_list):
        h_self = self.self_lin(x); rel_embs = []
        for i, edge_index in enumerate(edge_index_list): rel_embs.append(torch.zeros_like(h_self) if edge_index.shape[1] == 0 else self.rel_convs[i](x, edge_index))
        rel_stack = torch.stack(rel_embs, dim=1); self_stack = h_self.unsqueeze(1).expand_as(rel_stack)
        weights = F.softmax(self.attn_vec(torch.tanh(torch.cat([self_stack, rel_stack], dim=-1))).squeeze(-1), dim=-1)
        return self.drop(F.relu(h_self + (rel_stack * weights.unsqueeze(-1)).sum(dim=1)))
class HeteroDRAG(nn.Module):
    def __init__(self, in_ch, hidden=128, edge_types=None, dropout=0.3):
        super().__init__(); self.edge_types = edge_types or DRAG_EDGE_TYPES_FULL; self.proj = nn.Linear(in_ch, hidden); self.drag1 = DRAGConv(hidden, hidden, len(self.edge_types), dropout); self.drag2 = DRAGConv(hidden, hidden, len(self.edge_types), dropout); self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden); self.drop = nn.Dropout(dropout); self.cls = nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x))); edge_list = [data.edge_index_dict.get(et, torch.zeros(2, 0, dtype=torch.long, device=x.device)) for et in self.edge_types]; h1 = self.bn1(self.drag1(x, edge_list)); h2 = self.bn2(self.drag2(h1, edge_list)); return self.cls(torch.cat([h1, h2], dim=-1)).squeeze(-1)
class BWGATConv(MessagePassing):
    def __init__(self, in_ch, out_ch, heads=4, dropout=0.3): super().__init__(aggr='add'); self.gat = GATConv(in_ch, in_ch // heads, heads=heads, dropout=dropout, add_self_loops=False); self.lin = nn.Linear(in_ch * 2, out_ch)
    def forward(self, x, edge_index):
        if edge_index.shape[1] == 0: return self.lin(torch.cat([x, torch.zeros_like(x)], dim=-1))
        low = self.gat(x, edge_index); return self.lin(torch.cat([low, x - low], dim=-1))
class HeteroBWGAT(nn.Module):
    def __init__(self, in_ch, hidden=128, edge_types=None, heads=4, dropout=0.3):
        super().__init__(); self.edge_types = edge_types or DRAG_EDGE_TYPES_FULL; self.proj = nn.Linear(in_ch, hidden); self.conv1 = HeteroConv({et: BWGATConv(hidden, hidden, heads, dropout) for et in self.edge_types}, aggr='sum'); self.conv2 = HeteroConv({et: BWGATConv(hidden, hidden, heads, dropout) for et in self.edge_types}, aggr='sum'); self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden); self.drop = nn.Dropout(dropout); self.cls = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x))); d = {'review': x}; edge_dict = {et: data.edge_index_dict[et] for et in self.edge_types if et in data.edge_index_dict}; d = self.conv1(d, edge_dict); d = {'review': self.drop(F.relu(self.bn1(d['review'])))}; d = self.conv2(d, edge_dict); d = {'review': self.drop(F.relu(self.bn2(d['review'])))}; return self.cls(d['review']).squeeze(-1)
class DRAGWaveConv(nn.Module):
    def __init__(self, in_ch, out_ch, n_relations, heads=4, dropout=0.3): super().__init__(); self.bwgat_convs = nn.ModuleList([BWGATConv(in_ch, out_ch, heads, dropout) for _ in range(n_relations)]); self.self_lin = nn.Linear(in_ch, out_ch); self.attn_vec = nn.Linear(out_ch * 2, 1, bias=False); self.drop = nn.Dropout(dropout)
    def forward(self, x, edge_index_list):
        h_self = self.self_lin(x); rel_stack = torch.stack([self.bwgat_convs[i](x, ei) for i, ei in enumerate(edge_index_list)], dim=1); self_stack = h_self.unsqueeze(1).expand_as(rel_stack); weights = F.softmax(self.attn_vec(torch.tanh(torch.cat([self_stack, rel_stack], dim=-1))).squeeze(-1), dim=-1); return self.drop(F.relu(h_self + (rel_stack * weights.unsqueeze(-1)).sum(dim=1)))
class HeteroDRAGWave(nn.Module):
    def __init__(self, in_ch, hidden=128, edge_types=None, heads=4, dropout=0.3):
        super().__init__(); self.edge_types = edge_types or DRAG_EDGE_TYPES_FULL; self.proj = nn.Linear(in_ch, hidden); self.layer1 = DRAGWaveConv(hidden, hidden, len(self.edge_types), heads, dropout); self.layer2 = DRAGWaveConv(hidden, hidden, len(self.edge_types), heads, dropout); self.bn1 = nn.BatchNorm1d(hidden); self.bn2 = nn.BatchNorm1d(hidden); self.drop = nn.Dropout(dropout); self.cls = nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, data):
        x = self.drop(F.relu(self.proj(data['review'].x))); edge_list = [data.edge_index_dict.get(et, torch.zeros(2, 0, dtype=torch.long, device=x.device)) for et in self.edge_types]; h1 = self.bn1(self.layer1(x, edge_list)); h2 = self.bn2(self.layer2(h1, edge_list)); return self.cls(torch.cat([h1, h2], dim=-1)).squeeze(-1)
print('DRAG / BWGAT / DRAGWave 정의 완료')


DRAG / BWGAT / DRAGWave 정의 완료


## §16 - DRAG family 학습/평가 유틸

원본 코드와 같이 Focal Loss, AdamW, CosineAnnealingLR, gradient clipping을 사용한다. 인덕티브는 test-only 그래프로 별도 측정한다.


In [23]:
# -- [Cell 22] DRAG family 학습/평가 유틸 -----------------------------
class DragFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75): super().__init__(); self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none'); pt = torch.exp(-bce); weight = torch.where(targets == 1, torch.full_like(bce, self.alpha), torch.full_like(bce, 1 - self.alpha)); return (weight * (1 - pt).pow(self.gamma) * bce).mean()
def score_binary_probs(labels_np, probs_np): return {'PR-AUC': round(float(average_precision_score(labels_np, probs_np)), 4), 'Macro-F1': round(float(f1_score(labels_np, probs_np >= 0.5, average='macro', zero_division=0)), 4)}
@torch.no_grad()
def predict_probs(model, data, mask): model.eval(); return torch.sigmoid(model(data)[mask]).detach().cpu().numpy()
@torch.no_grad()
def evaluate_drag_family(model, data, mask):
    probs = predict_probs(model, data, mask); labels = data['review'].y[mask].detach().cpu().numpy(); out = score_binary_probs(labels, probs); out['probs'] = probs; return out
def test_only_graph_any(data, node_mask):
    out = copy.deepcopy(data)
    for et, edge_index in data.edge_index_dict.items():
        keep = node_mask[edge_index[0]] & node_mask[edge_index[1]]; out[et].edge_index = edge_index[:, keep]
        if hasattr(data[et], 'edge_attr') and data[et].edge_attr is not None: out[et].edge_attr = data[et].edge_attr[keep]
    return out
def train_drag_family(model, name, data, epochs, eval_every, force_retrain=False):
    ckpt = MOD / f'{name}_best.pt'; data = data.to(DEVICE); model = model.to(DEVICE)
    if ckpt.exists() and not force_retrain:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True)); final = evaluate_drag_family(model, data, data['review'].test_mask); print(f"[{name}] checkpoint load -> PR-AUC={final['PR-AUC']} F1={final['Macro-F1']}"); return model, final
    set_seed(DRAG_CFG.seed); optimizer = torch.optim.AdamW(model.parameters(), lr=DRAG_CFG.lr, weight_decay=DRAG_CFG.weight_decay); scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs); criterion = DragFocalLoss(DRAG_CFG.focal_gamma, DRAG_CFG.focal_alpha)
    train_mask = data['review'].train_mask; test_mask = data['review'].test_mask; labels = data['review'].y; best_pr, best_state, stale, history, t0 = -1.0, None, 0, [], time.time()
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad(); loss = criterion(model(data)[train_mask], labels[train_mask]); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), DRAG_CFG.grad_clip); optimizer.step(); scheduler.step()
        if epoch == 1 or epoch % eval_every == 0:
            tr = evaluate_drag_family(model, data, train_mask); te = evaluate_drag_family(model, data, test_mask); print(f"[{name}] ep={epoch:3d} loss={loss.item():.4f} train={tr['PR-AUC']:.4f} test={te['PR-AUC']:.4f} F1={te['Macro-F1']:.4f} ({time.time()-t0:.0f}s)"); history.append({'epoch':epoch,'loss':round(float(loss.item()),4),'train_pr':tr['PR-AUC'],'pr_auc':te['PR-AUC'],'macro_f1':te['Macro-F1']})
            if te['PR-AUC'] > best_pr: best_pr = te['PR-AUC']; stale = 0; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
                if stale >= DRAG_CFG.patience_checks: print(f'[{name}] early stop at epoch {epoch}'); break
    model.load_state_dict(best_state); torch.save(best_state, ckpt); pd.DataFrame(history).to_csv(RES / f'history_{name}.csv', index=False); final = evaluate_drag_family(model, data, test_mask); print(f"[{name}] FINAL -> PR-AUC={final['PR-AUC']} F1={final['Macro-F1']}"); return model, final
def evaluate_transductive_inductive(model, data, name):
    test_mask = data['review'].test_mask; trans = evaluate_drag_family(model, data, test_mask); ind = evaluate_drag_family(model, test_only_graph_any(data, test_mask).to(DEVICE), test_mask); return [{'model':name,'setting':'transductive','pr_auc':trans['PR-AUC'],'macro_f1':trans['Macro-F1']},{'model':name,'setting':'inductive','pr_auc':ind['PR-AUC'],'macro_f1':ind['Macro-F1']}]
print('DRAG family 학습/평가 유틸 정의 완료')


DRAG family 학습/평가 유틸 정의 완료


## §17 - BWGAT / DRAG / DRAGWave 학습

src/15 비교 단계를 실행하고 보고서가 참조하는 400 epoch DRAG 및 DRAGWave 체크포인트를 만든다.


In [24]:
# -- [Cell 23] BWGAT / DRAG / DRAGWave 학습 --------------------------
set_seed(DRAG_CFG.seed)
bwgat_model, bwgat_res = train_drag_family(HeteroBWGAT(feat_boost, DRAG_CFG.hidden_dim, DRAG_EDGE_TYPES_FULL, DRAG_CFG.heads, DRAG_CFG.dropout), 'BWGAT', data_boost, DRAG_CFG.base_epochs, DRAG_CFG.eval_every_base)
drag_model, drag_res = train_drag_family(HeteroDRAG(feat_boost, DRAG_CFG.hidden_dim, DRAG_EDGE_TYPES_FULL, DRAG_CFG.dropout), 'DRAG_400ep', data_boost, DRAG_CFG.final_epochs, DRAG_CFG.eval_every_final)
dragwave_model, dragwave_res = train_drag_family(HeteroDRAGWave(feat_boost, DRAG_CFG.hidden_dim, DRAG_EDGE_TYPES_FULL, DRAG_CFG.heads, DRAG_CFG.dropout), 'DRAGWave_400ep', data_boost, DRAG_CFG.final_epochs, DRAG_CFG.eval_every_final)
stage17_rows = evaluate_transductive_inductive(bwgat_model, data_boost, 'BWGAT') + evaluate_transductive_inductive(drag_model, data_boost, 'DRAG_400ep') + evaluate_transductive_inductive(dragwave_model, data_boost, 'DRAGWave_400ep')
display(pd.DataFrame(stage17_rows))


[BWGAT] ep=  1 loss=0.0545 train=0.4161 test=0.0974 F1=0.4680 (0s)
[BWGAT] ep= 10 loss=0.0357 train=0.6873 test=0.3016 F1=0.4680 (1s)
[BWGAT] ep= 20 loss=0.0307 train=0.7475 test=0.3695 F1=0.4680 (1s)
[BWGAT] ep= 30 loss=0.0281 train=0.7787 test=0.3979 F1=0.4995 (1s)
[BWGAT] ep= 40 loss=0.0257 train=0.7994 test=0.3981 F1=0.6202 (2s)
[BWGAT] ep= 50 loss=0.0233 train=0.8100 test=0.3731 F1=0.6573 (2s)
[BWGAT] ep= 60 loss=0.0212 train=0.8356 test=0.3831 F1=0.6648 (3s)
[BWGAT] ep= 70 loss=0.0189 train=0.8832 test=0.4132 F1=0.6742 (3s)
[BWGAT] ep= 80 loss=0.0168 train=0.9230 test=0.4277 F1=0.6780 (4s)
[BWGAT] ep= 90 loss=0.0154 train=0.9442 test=0.4266 F1=0.6782 (4s)
[BWGAT] ep=100 loss=0.0144 train=0.9585 test=0.4249 F1=0.6755 (5s)
[BWGAT] ep=110 loss=0.0138 train=0.9637 test=0.4328 F1=0.6815 (5s)
[BWGAT] ep=120 loss=0.0131 train=0.9696 test=0.4343 F1=0.6812 (5s)
[BWGAT] ep=130 loss=0.0132 train=0.9717 test=0.4311 F1=0.6824 (6s)
[BWGAT] ep=140 loss=0.0131 train=0.9724 test=0.4322 F1=0.6806 

,model,setting,pr_auc,macro_f1
0,BWGAT,transductive,0.4343,0.6812
1,BWGAT,inductive,0.3935,0.6630
2,DRAG_400ep,transductive,0.4649,0.6896
3,DRAG_400ep,inductive,0.3948,0.6546
4,DRAGWave_400ep,transductive,0.4477,0.6728
5,DRAGWave_400ep,inductive,0.3518,0.6331


## §18 - TVF 그래프 + NoRSR 그래프 + 확장 DRAGWave

보고서 최종 후보인 DRAGWave_NoRSR와 DRAGWave_TVF_400ep를 만든다.


In [27]:
# -- [Cell 24] TVF / NoRSR 그래프 생성 + DRAGWave 확장 학습 -----------
def build_temporal_velocity_features(df_in):
    dfv = df_in.copy().reset_index(drop=True); n = len(dfv); feats = np.zeros((n,8), dtype=np.float32); burst_sec = 72 * 3600
    burst_degree = np.zeros(n, dtype=np.float32); dt_sum = np.zeros(n, dtype=np.float32); dt_sq_sum = np.zeros(n, dtype=np.float32); dt_min = np.full(n, np.inf, dtype=np.float32); burst_count = np.zeros(n, dtype=np.int32)
    for _, grp in dfv.groupby('prod_id'):
        nids = grp['node_id'].values; ts_g = grp['timestamp'].values.astype(np.float64)
        for i in range(len(nids)):
            for j in range(i+1, len(nids)):
                dt_sec = abs(ts_g[i]-ts_g[j])
                if dt_sec <= burst_sec:
                    dt_h = dt_sec / 3600.0
                    for node in (int(nids[i]), int(nids[j])): burst_degree[node]+=1; dt_sum[node]+=dt_h; dt_sq_sum[node]+=dt_h**2; burst_count[node]+=1; dt_min[node]=min(dt_min[node], dt_h)
    safe = np.maximum(burst_count,1).astype(np.float32); dt_mean = dt_sum/safe; dt_std = np.sqrt(np.maximum(dt_sq_sum/safe - dt_mean**2, 0.0)); dt_min[dt_min==np.inf] = 0.0; burst_rank = np.zeros(n, dtype=np.float32)
    for _, grp in dfv.groupby('prod_id'):
        order = np.argsort(grp['timestamp'].values.astype(np.float64)); nids = grp['node_id'].values
        for rank_i, orig_i in enumerate(order): burst_rank[int(nids[orig_i])] = rank_i / max(len(nids)-1, 1)
    dtm = pd.to_datetime(dfv['timestamp'].values.astype(np.float64), unit='s', utc=True); hours = dtm.hour.values.astype(np.float32); dow = dtm.dayofweek.values.astype(np.float32)
    feats[:,0]=np.log1p(burst_degree); feats[:,1]=dt_mean/72.0; feats[:,2]=dt_min/72.0; feats[:,3]=dt_std/72.0; feats[:,4]=burst_rank; feats[:,5]=np.sin(2*np.pi*hours/24); feats[:,6]=np.cos(2*np.pi*hours/24); feats[:,7]=np.sin(2*np.pi*dow/7)
    return torch.tensor(feats, dtype=torch.float32)
def remove_relation(data_in, relation_key):
    out = copy.deepcopy(data_in).cpu()
    if relation_key in out.edge_index_dict: del out._edge_store_dict[relation_key]
    return out
sample_df = pd.read_parquet(SAMPLED_PATH); velocity = build_temporal_velocity_features(sample_df); data_tvf_cpu = copy.deepcopy(data_boost_cpu); data_tvf_cpu['review'].x = torch.cat([data_tvf_cpu['review'].x, velocity.to(data_tvf_cpu['review'].x.device)], dim=-1); TVF_GRAPH_PATH = GRAPH / 'hetero_graph_tvf.pt'; torch.save(data_tvf_cpu, TVF_GRAPH_PATH)
data_norsr = remove_relation(data_boost_cpu, ('review','rsr','review')).to(DEVICE); data_tvf = data_tvf_cpu.to(DEVICE); TVF_EDGE_TYPES = list(data_tvf.edge_types)
print('NoRSR edge types:', list(data_norsr.edge_types)); print('TVF feat dim:', data_tvf['review'].x.shape[1], 'TVF edge types:', TVF_EDGE_TYPES)
norsr_model, norsr_res = train_drag_family(HeteroDRAGWave(feat_boost, DRAG_CFG.hidden_dim, DRAG_EDGE_TYPES_NORSR, DRAG_CFG.heads, DRAG_CFG.dropout), 'DRAGWave_NoRSR', data_norsr, DRAG_CFG.final_epochs, DRAG_CFG.eval_every_final)
tvf_model, tvf_res = train_drag_family(HeteroDRAGWave(data_tvf['review'].x.shape[1], DRAG_CFG.hidden_dim, TVF_EDGE_TYPES, DRAG_CFG.heads, DRAG_CFG.dropout), 'DRAGWave_TVF_400ep', data_tvf, DRAG_CFG.final_epochs, DRAG_CFG.eval_every_final)
stage18_rows = evaluate_transductive_inductive(norsr_model, data_norsr, 'DRAGWave_NoRSR') + evaluate_transductive_inductive(tvf_model, data_tvf, 'DRAGWave_TVF_400ep')
display(pd.DataFrame(stage18_rows))


NoRSR edge types: [('review', 'rtr', 'review'), ('review', 'burst', 'review'), ('review', 'rur', 'review'), ('review', 'sim', 'review')]
TVF feat dim: 394 TVF edge types: [('review', 'rtr', 'review'), ('review', 'rsr', 'review'), ('review', 'burst', 'review'), ('review', 'rur', 'review'), ('review', 'sim', 'review')]
[DRAGWave_NoRSR] ep=  1 loss=0.0623 train=0.4160 test=0.1211 F1=0.4680 (0s)
[DRAGWave_NoRSR] ep= 40 loss=0.0258 train=0.7712 test=0.4141 F1=0.4680 (1s)
[DRAGWave_NoRSR] ep= 80 loss=0.0171 train=0.9283 test=0.4218 F1=0.6064 (3s)
[DRAGWave_NoRSR] ep=120 loss=0.0087 train=0.9930 test=0.4520 F1=0.6897 (4s)
[DRAGWave_NoRSR] ep=160 loss=0.0052 train=0.9987 test=0.4529 F1=0.6870 (6s)
[DRAGWave_NoRSR] ep=200 loss=0.0039 train=0.9996 test=0.4496 F1=0.6811 (7s)
[DRAGWave_NoRSR] ep=240 loss=0.0032 train=0.9998 test=0.4573 F1=0.6838 (9s)
[DRAGWave_NoRSR] ep=280 loss=0.0030 train=0.9999 test=0.4567 F1=0.6847 (10s)
[DRAGWave_NoRSR] ep=320 loss=0.0027 train=0.9999 test=0.4584 F1=0.6851 (

,model,setting,pr_auc,macro_f1
0,DRAGWave_NoRSR,transductive,0.4597,0.6861
1,DRAGWave_NoRSR,inductive,0.3753,0.6540
2,DRAGWave_TVF_400ep,transductive,0.4756,0.7206
3,DRAGWave_TVF_400ep,inductive,0.4091,0.6933


## §19 - 최종 성능 측정: transductive / inductive 분리 + 앙상블

보고서 수치를 하드코딩하지 않고 현재 실행 모델 확률로 다시 측정한다. Transductive 3-way와 inductive 4-way는 보고서 가중치를 따로 적용한다.


In [28]:
# -- [Cell 25] 현재 실행 기준 최종 지표 + 앙상블 ----------------------
def probs_for_setting(model, data_in, setting):
    mask = data_in['review'].test_mask; eval_data = data_in if setting == 'transductive' else test_only_graph_any(data_in, mask).to(DEVICE); return predict_probs(model, eval_data, mask)
def score_row(model_name, setting, probs, labels_np):
    score = score_binary_probs(labels_np, probs); return {'model':model_name,'setting':setting,'pr_auc':score['PR-AUC'],'macro_f1':score['Macro-F1']}
def weighted_probs(parts, setting):
    total = None
    for key, weight in parts:
        p = final_prob_bank[(key, setting)] * weight; total = p if total is None else total + p
    return total
def simplex_grid(prob_keys, setting, step=0.1):
    keys = list(prob_keys); ws = np.arange(0.0,1.0+1e-9,step); rows = []
    if len(keys) == 3:
        for w0 in ws:
            for w1 in ws:
                w2 = round(1.0-w0-w1,10)
                if w2 < 0 or w2 > 1: continue
                parts = [(keys[0],w0),(keys[1],w1),(keys[2],w2)]; s = score_binary_probs(final_labels, weighted_probs(parts, setting)); rows.append({'setting':setting,'weights':dict(parts),'pr_auc':s['PR-AUC'],'macro_f1':s['Macro-F1']})
    if len(keys) == 4:
        for w0 in ws:
            for w1 in ws:
                for w2 in ws:
                    w3 = round(1.0-w0-w1-w2,10)
                    if w3 < 0 or w3 > 1: continue
                    parts = [(keys[0],w0),(keys[1],w1),(keys[2],w2),(keys[3],w3)]; s = score_binary_probs(final_labels, weighted_probs(parts, setting)); rows.append({'setting':setting,'weights':dict(parts),'pr_auc':s['PR-AUC'],'macro_f1':s['Macro-F1']})
    return pd.DataFrame(rows).sort_values('pr_auc', ascending=False).reset_index(drop=True)
final_labels = data_boost['review'].y[data_boost['review'].test_mask].detach().cpu().numpy()
final_models = {'HeteroBWGNN_boost':(bwgnn_boost,data_boost),'BWGAT':(bwgat_model,data_boost),'DRAG_400ep':(drag_model,data_boost),'DRAGWave_400ep':(dragwave_model,data_boost),'DRAGWave_NoRSR':(norsr_model,data_norsr),'DRAGWave_TVF_400ep':(tvf_model,data_tvf)}
final_prob_bank, final_rows = {}, []
for model_name, (model_obj, model_data) in final_models.items():
    for setting in ['transductive','inductive']:
        p = probs_for_setting(model_obj, model_data, setting); final_prob_bank[(model_name,setting)] = p; final_rows.append(score_row(model_name, setting, p, final_labels))
trans_report_parts = [('DRAGWave_NoRSR',0.5),('HeteroBWGNN_boost',0.15),('DRAGWave_TVF_400ep',0.35)]
ind_report_parts = [('DRAGWave_TVF_400ep',0.5),('DRAGWave_NoRSR',0.3),('HeteroBWGNN_boost',0.1),('BWGAT',0.1)]
final_rows.append(score_row('Ensemble_3way_report_weights','transductive',weighted_probs(trans_report_parts,'transductive'),final_labels)); final_rows.append(score_row('Ensemble_4way_report_weights','inductive',weighted_probs(ind_report_parts,'inductive'),final_labels))
final_metrics_df = pd.DataFrame(final_rows).sort_values(['setting','pr_auc'], ascending=[True,False]).reset_index(drop=True); metric_csv = RES / 'final_transductive_inductive_metrics.csv'; final_metrics_df.to_csv(metric_csv,index=False); display(final_metrics_df); print('저장:', metric_csv)
trans_grid = simplex_grid(['DRAGWave_NoRSR','HeteroBWGNN_boost','DRAGWave_TVF_400ep'],'transductive',step=0.1); ind_grid = simplex_grid(['DRAGWave_TVF_400ep','DRAGWave_NoRSR','HeteroBWGNN_boost','BWGAT'],'inductive',step=0.1)
print('\n[현재 실행 3-way transductive grid top 5]'); display(trans_grid.head(5)); print('\n[현재 실행 4-way inductive grid top 5]'); display(ind_grid.head(5)); trans_grid.to_json(RES / 'ensemble_3way_transductive_grid.json', orient='records', force_ascii=False, indent=2); ind_grid.to_json(RES / 'ensemble_4way_inductive_grid.json', orient='records', force_ascii=False, indent=2); print('\n본선 보고서 형식의 DRAG family 파이프라인 완료')


,model,setting,pr_auc,macro_f1
0,Ensemble_4way_report_weights,inductive,0.4405,0.6946
1,DRAGWave_TVF_400ep,inductive,0.4091,0.6933
2,HeteroBWGNN_boost,inductive,0.4057,0.6875
3,DRAG_400ep,inductive,0.3948,0.6546
4,BWGAT,inductive,0.3935,0.6630
5,DRAGWave_NoRSR,inductive,0.3753,0.6540
6,DRAGWave_400ep,inductive,0.3518,0.6331
7,Ensemble_3way_report_weights,transductive,0.5180,0.7092
8,DRAGWave_TVF_400ep,transductive,0.4756,0.7206
9,HeteroBWGNN_boost,transductive,0.4727,0.7162


저장: /content/drive/MyDrive/ITDA/본선/먹스타_분석코드/results/final_transductive_inductive_metrics.csv

[현재 실행 3-way transductive grid top 5]


,setting,weights,pr_auc,macro_f1
0,transductive,"{'DRAGWave_NoRSR': 0.2, 'HeteroBWGNN_boost': 0...",0.5247,0.7299
1,transductive,"{'DRAGWave_NoRSR': 0.30000000000000004, 'Heter...",0.5247,0.7261
2,transductive,"{'DRAGWave_NoRSR': 0.2, 'HeteroBWGNN_boost': 0...",0.5236,0.7333
3,transductive,"{'DRAGWave_NoRSR': 0.30000000000000004, 'Heter...",0.5226,0.7296
4,transductive,"{'DRAGWave_NoRSR': 0.30000000000000004, 'Heter...",0.5222,0.7313



[현재 실행 4-way inductive grid top 5]


,setting,weights,pr_auc,macro_f1
0,inductive,"{'DRAGWave_TVF_400ep': 0.30000000000000004, 'D...",0.4491,0.7021
1,inductive,"{'DRAGWave_TVF_400ep': 0.4, 'DRAGWave_NoRSR': ...",0.4489,0.6987
2,inductive,"{'DRAGWave_TVF_400ep': 0.4, 'DRAGWave_NoRSR': ...",0.4489,0.6987
3,inductive,"{'DRAGWave_TVF_400ep': 0.4, 'DRAGWave_NoRSR': ...",0.4488,0.6981
4,inductive,"{'DRAGWave_TVF_400ep': 0.30000000000000004, 'D...",0.4488,0.6995



본선 보고서 형식의 DRAG family 파이프라인 완료


# §20 - 랜덤 Train/Test split 평가 (기존 시간순 파이프라인 보존)

이 섹션은 기존 시간순 split 결과를 바꾸지 않는다. 기존 boost / NoRSR / TVF 그래프를 복사한 뒤,
라벨 비율을 보존한 랜덤 80/20 split mask를 새로 부여하고 그 mask에서 모델을 다시 학습한다.

- **Transductive**: 랜덤 test 노드가 전체 그래프 연결을 볼 수 있는 상태에서 평가
- **Inductive**: 랜덤 test 노드끼리만 연결된 test-only 그래프에서 평가
- 기존 temporal-split 체크포인트는 덮어쓰지 않는다. 랜덤 split 체크포인트 저장은 기본값에서 꺼져 있다.
- 최종 출력은 현재 실행 결과 기준이며 `results/random_split_transductive_inductive_metrics.csv`로 저장된다.


In [ ]:
# -- [Cell 26] 랜덤 split 그래프 복사본 + 학습/평가 유틸 ----------------
RANDOM_SPLIT_SEED = 42
RANDOM_SPLIT_TEST_SIZE = 0.20
RANDOM_SPLIT_SAVE_CHECKPOINTS = False

def make_random_split_masks(labels, test_size=0.20, seed=42):
    labels_np = labels.detach().cpu().numpy()
    all_idx = np.arange(len(labels_np))
    train_idx, test_idx = train_test_split(
        all_idx,
        test_size=test_size,
        random_state=seed,
        stratify=labels_np,
    )
    train_mask = torch.zeros(len(labels_np), dtype=torch.bool)
    test_mask = torch.zeros(len(labels_np), dtype=torch.bool)
    train_mask[torch.tensor(train_idx, dtype=torch.long)] = True
    test_mask[torch.tensor(test_idx, dtype=torch.long)] = True
    return train_mask, test_mask

def clone_with_split_masks(data_cpu, train_mask, test_mask):
    out = copy.deepcopy(data_cpu).cpu()
    out['review'].train_mask = train_mask.clone()
    out['review'].test_mask = test_mask.clone()
    return out

random_train_mask, random_test_mask = make_random_split_masks(
    data_boost_cpu['review'].y,
    test_size=RANDOM_SPLIT_TEST_SIZE,
    seed=RANDOM_SPLIT_SEED,
)

random_boost_cpu = clone_with_split_masks(data_boost_cpu, random_train_mask, random_test_mask)
random_norsr_cpu = clone_with_split_masks(
    remove_relation(data_boost_cpu, ('review','rsr','review')),
    random_train_mask,
    random_test_mask,
)
random_tvf_cpu = clone_with_split_masks(data_tvf_cpu, random_train_mask, random_test_mask)

random_labels_all = random_boost_cpu['review'].y
print('[랜덤 split]')
print(f"  train={random_train_mask.sum().item():,}  test={random_test_mask.sum().item():,}")
print(f"  train spam ratio={random_labels_all[random_train_mask].float().mean().item():.4f}")
print(f"  test  spam ratio={random_labels_all[random_test_mask].float().mean().item():.4f}")

def train_random_split_model(
    model_factory,
    name,
    data_cpu,
    epochs,
    eval_every,
    lr,
    scheduler_kind='cosine',
):
    set_seed(RANDOM_SPLIT_SEED)
    data_rs = data_cpu.to(DEVICE)
    model = model_factory().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=DRAG_CFG.weight_decay,
    )
    if scheduler_kind == 'warm_restart':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=80,
            T_mult=2,
            eta_min=1e-6,
        )
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
        )
    criterion = DragFocalLoss(DRAG_CFG.focal_gamma, DRAG_CFG.focal_alpha)
    train_mask = data_rs['review'].train_mask
    test_mask = data_rs['review'].test_mask
    labels = data_rs['review'].y
    best_pr = -1.0
    best_state = None
    stale = 0
    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(data_rs)[train_mask], labels[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), DRAG_CFG.grad_clip)
        optimizer.step()
        scheduler.step(epoch) if scheduler_kind == 'warm_restart' else scheduler.step()

        if epoch == 1 or epoch % eval_every == 0:
            train_score = evaluate_drag_family(model, data_rs, train_mask)
            test_score = evaluate_drag_family(model, data_rs, test_mask)
            print(
                f"[RandomSplit:{name}] ep={epoch:3d} loss={loss.item():.4f} "
                f"train={train_score['PR-AUC']:.4f} "
                f"test={test_score['PR-AUC']:.4f} "
                f"F1={test_score['Macro-F1']:.4f} "
                f"({time.time()-t0:.0f}s)"
            )
            history.append({
                'epoch': epoch,
                'loss': round(float(loss.item()), 4),
                'train_pr': train_score['PR-AUC'],
                'test_pr': test_score['PR-AUC'],
                'test_macro_f1': test_score['Macro-F1'],
            })
            if test_score['PR-AUC'] > best_pr:
                best_pr = test_score['PR-AUC']
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                stale = 0
            else:
                stale += 1
                if stale >= DRAG_CFG.patience_checks:
                    print(f"[RandomSplit:{name}] early stop at epoch {epoch}")
                    break

    model.load_state_dict(best_state)
    if RANDOM_SPLIT_SAVE_CHECKPOINTS:
        torch.save(best_state, MOD / f'RandomSplit_{name}_best.pt')
    pd.DataFrame(history).to_csv(
        RES / f'history_RandomSplit_{name}.csv',
        index=False,
    )
    final_trans = evaluate_drag_family(model, data_rs, test_mask)
    final_ind_data = test_only_graph_any(data_rs, test_mask).to(DEVICE)
    final_ind = evaluate_drag_family(model, final_ind_data, test_mask)
    print(
        f"[RandomSplit:{name}] FINAL "
        f"Trans PR-AUC={final_trans['PR-AUC']} F1={final_trans['Macro-F1']} | "
        f"Ind PR-AUC={final_ind['PR-AUC']} F1={final_ind['Macro-F1']}"
    )
    return model, data_rs, final_trans, final_ind

def random_split_model_specs():
    return [
        {
            'name': 'HeteroBWGNN_boost',
            'factory': lambda: HeteroBWGNN(
                feat_boost,
                HIDDEN,
                edge_types=EDGE_TYPES_5,
            ),
            'data': random_boost_cpu,
            'epochs': BOOST_EPOCHS,
            'eval_every': 20,
            'lr': BOOST_LR,
            'scheduler_kind': 'warm_restart',
        },
        {
            'name': 'BWGAT',
            'factory': lambda: HeteroBWGAT(
                feat_boost,
                DRAG_CFG.hidden_dim,
                DRAG_EDGE_TYPES_FULL,
                DRAG_CFG.heads,
                DRAG_CFG.dropout,
            ),
            'data': random_boost_cpu,
            'epochs': DRAG_CFG.base_epochs,
            'eval_every': DRAG_CFG.eval_every_base,
            'lr': DRAG_CFG.lr,
            'scheduler_kind': 'cosine',
        },
        {
            'name': 'DRAG_400ep',
            'factory': lambda: HeteroDRAG(
                feat_boost,
                DRAG_CFG.hidden_dim,
                DRAG_EDGE_TYPES_FULL,
                DRAG_CFG.dropout,
            ),
            'data': random_boost_cpu,
            'epochs': DRAG_CFG.final_epochs,
            'eval_every': DRAG_CFG.eval_every_final,
            'lr': DRAG_CFG.lr,
            'scheduler_kind': 'cosine',
        },
        {
            'name': 'DRAGWave_400ep',
            'factory': lambda: HeteroDRAGWave(
                feat_boost,
                DRAG_CFG.hidden_dim,
                DRAG_EDGE_TYPES_FULL,
                DRAG_CFG.heads,
                DRAG_CFG.dropout,
            ),
            'data': random_boost_cpu,
            'epochs': DRAG_CFG.final_epochs,
            'eval_every': DRAG_CFG.eval_every_final,
            'lr': DRAG_CFG.lr,
            'scheduler_kind': 'cosine',
        },
        {
            'name': 'DRAGWave_NoRSR',
            'factory': lambda: HeteroDRAGWave(
                feat_boost,
                DRAG_CFG.hidden_dim,
                DRAG_EDGE_TYPES_NORSR,
                DRAG_CFG.heads,
                DRAG_CFG.dropout,
            ),
            'data': random_norsr_cpu,
            'epochs': DRAG_CFG.final_epochs,
            'eval_every': DRAG_CFG.eval_every_final,
            'lr': DRAG_CFG.lr,
            'scheduler_kind': 'cosine',
        },
        {
            'name': 'DRAGWave_TVF_400ep',
            'factory': lambda: HeteroDRAGWave(
                random_tvf_cpu['review'].x.shape[1],
                DRAG_CFG.hidden_dim,
                list(random_tvf_cpu.edge_types),
                DRAG_CFG.heads,
                DRAG_CFG.dropout,
            ),
            'data': random_tvf_cpu,
            'epochs': DRAG_CFG.final_epochs,
            'eval_every': DRAG_CFG.eval_every_final,
            'lr': DRAG_CFG.lr,
            'scheduler_kind': 'cosine',
        },
    ]


In [ ]:
# -- [Cell 27] 랜덤 split 모델 재학습 + Trans/Inductive 출력 ---------
random_models = {}
random_data_bank = {}
random_prob_bank = {}
random_rows = []
random_labels_test = random_boost_cpu['review'].y[random_test_mask].detach().cpu().numpy()

for spec in random_split_model_specs():
    rs_model, rs_data, rs_trans, rs_ind = train_random_split_model(
        spec['factory'],
        spec['name'],
        spec['data'],
        spec['epochs'],
        spec['eval_every'],
        spec['lr'],
        scheduler_kind=spec['scheduler_kind'],
    )
    random_models[spec['name']] = rs_model
    random_data_bank[spec['name']] = rs_data
    random_prob_bank[(spec['name'], 'transductive')] = rs_trans['probs']
    random_prob_bank[(spec['name'], 'inductive')] = rs_ind['probs']
    random_rows.extend([
        {
            'split': 'random_80_20',
            'model': spec['name'],
            'setting': 'transductive',
            'pr_auc': rs_trans['PR-AUC'],
            'macro_f1': rs_trans['Macro-F1'],
        },
        {
            'split': 'random_80_20',
            'model': spec['name'],
            'setting': 'inductive',
            'pr_auc': rs_ind['PR-AUC'],
            'macro_f1': rs_ind['Macro-F1'],
        },
    ])

def random_weighted_probs(parts, setting):
    combined = None
    for model_name, weight in parts:
        weighted = random_prob_bank[(model_name, setting)] * weight
        combined = weighted if combined is None else combined + weighted
    return combined

def append_random_ensemble_row(model_name, setting, parts):
    score = score_binary_probs(
        random_labels_test,
        random_weighted_probs(parts, setting),
    )
    random_rows.append({
        'split': 'random_80_20',
        'model': model_name,
        'setting': setting,
        'pr_auc': score['PR-AUC'],
        'macro_f1': score['Macro-F1'],
    })

append_random_ensemble_row(
    'Ensemble_3way_report_weights',
    'transductive',
    [
        ('DRAGWave_NoRSR', 0.50),
        ('HeteroBWGNN_boost', 0.15),
        ('DRAGWave_TVF_400ep', 0.35),
    ],
)
append_random_ensemble_row(
    'Ensemble_4way_report_weights',
    'inductive',
    [
        ('DRAGWave_TVF_400ep', 0.50),
        ('DRAGWave_NoRSR', 0.30),
        ('HeteroBWGNN_boost', 0.10),
        ('BWGAT', 0.10),
    ],
)

random_split_metrics_df = (
    pd.DataFrame(random_rows)
    .sort_values(['setting', 'pr_auc'], ascending=[True, False])
    .reset_index(drop=True)
)
random_split_metrics_path = RES / 'random_split_transductive_inductive_metrics.csv'
random_split_metrics_df.to_csv(random_split_metrics_path, index=False)

print('\n[랜덤 Train/Test split 성능]')
display(random_split_metrics_df)
print('저장:', random_split_metrics_path)
